In [1]:
from __future__ import annotations
import logging
import hashlib
import os

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal
from uuid import uuid4

from dotenv import load_dotenv
from pydantic import BaseModel, Field



In [2]:
# !rm -rf chroma_db/
# !rm -f data/processed/*.json
# !rm -f eval/eval_set.json
# !rm -rf eval/results/

In [3]:
## resolve project root directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")


True

In [4]:
PROJECT_ROOT

PosixPath('/home/thimu/github_vs/protoRAG/rag-pipeline')

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)-7s | %(name)s | %(message)s",
)
# Add after logging.basicConfig in Cell 1
logging.getLogger("transformers").setLevel(logging.CRITICAL)
log = logging.getLogger("rag")

In [6]:
def _path(env_key: str, default: str) -> Path:
    """Resolve env-provided paths relative to project root unless absolute."""
    p = Path(os.getenv(env_key, default))
    return p.resolve() if p.is_absolute() else (PROJECT_ROOT / p).resolve()

In [7]:
class Config:
    # ollama
    OLLAMA_HOST: str = os.getenv("OLLAMA_HOST", "localhost")
    OLLAMA_PORT: int = int(os.getenv("OLLAMA_PORT", 11434))
    OLLAMA_MODEL: str = os.getenv("OLLAMA_MODEL", "gemma-4-e4b:latest")
    EMBEDDING_MODEL: str = os.getenv("EMBEDDING_MODEL", "embeddinggemma:latest")

    @property
    def OLLAMA_BASE_URL(self) -> str:
        return f"http://{self.OLLAMA_HOST}:{self.OLLAMA_PORT}"
    
    # Generation
    LLM_TEMPERATURE: float = float(os.getenv("LLM_TEMPERATURE", 0.0))
    LLM_NUM_CTX: int = int(os.getenv("LLM_NUM_CTX", 8192))

    # Retrieval
    CHUNK_SIZE: int = int(os.getenv("CHUNK_SIZE", 800))
    CHUNK_OVERLAP: int = int(os.getenv("CHUNK_OVERLAP", 120))
    TOP_K: int = int(os.getenv("TOP_K", 5))

    # Paths:
    CHROMA_PERSIST_DIR: Path = _path("CHROMA_PERSIST_DIR", "chroma_db")
    DATA_RAW_DIR: Path = _path("DATA_RAW_DIR", "/home/thimu/Downloads/pdf_splitter/IPC")

    DATA_PROCESSED_DIR: Path = _path("DATA_PROCESSED_DIR", "data/processed")

cfg = Config()


In [8]:
cfg.CHROMA_PERSIST_DIR.mkdir(parents=True, exist_ok=True)
cfg.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
log.info(f"ollama base url: {cfg.OLLAMA_BASE_URL}")
log.info(f"ollama model: {cfg.OLLAMA_MODEL}")
log.info(f"embedding model: {cfg.EMBEDDING_MODEL}")
log.info(f"chunk/overlap/k: {cfg.CHUNK_SIZE}/{cfg.CHUNK_OVERLAP}/{cfg.TOP_K}")
log.info(f"Project root: {PROJECT_ROOT}")


2026-06-02 00:42:30,384 - INFO    | rag | ollama base url: http://localhost:11434
2026-06-02 00:42:30,385 - INFO    | rag | ollama model: gemma-4-e4b:latest
2026-06-02 00:42:30,386 - INFO    | rag | embedding model: embeddinggemma:latest
2026-06-02 00:42:30,386 - INFO    | rag | chunk/overlap/k: 800/120/5
2026-06-02 00:42:30,386 - INFO    | rag | Project root: /home/thimu/github_vs/protoRAG/rag-pipeline


In [10]:
SourceFormat = Literal["pdf", "txt", "md", "html", "docx", "xlsx", "pptx", "csv", "json", "xml", "jsonl", "yaml", "yml", "parquet", "avro", "orc", "tsv", "log"]
ElementType = Literal["text", "table", "list", "code", "heading", "metadata", "other"]

class RagChunk(BaseModel):
    """The atomic unit flowing through retrieval. All parsers produce these."""

    # identity
    chunk_id: str = Field(default_factory=lambda: str(uuid4()))
    content_hash: str = ""

    # content
    text: str

    # provenance - these are citation
    source_path: str
    source_format: SourceFormat

    # optional structured metadata (parser-specific, all optional)
    page_number: int | None = None # pdf
    slide_number: int | None = None # pptx
    sheet_name: str | None = None # excel
    section_heading: str | None = None # general document structure
    section_title: str | None = None # pdf bookmarks, html headings, etc.
    row_range: tuple[int, int] | None = None # csv, excel tables

    element_type: ElementType = "text"

    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    extra: dict[str, Any] = Field(default_factory=dict)

    def model_post_init(self, __context: Any) -> None:
        if not self.content_hash:
            self.content_hash = hashlib.sha256(self.text.encode("utf-8")).hexdigest()[:16]

    def to_langchain_metadata(self) -> dict[str, Any]:
        """Flat, JSON-safe, no-None dict for Chroma / LangChain Document.metadata"""
        md: dict[str, Any] = {
            "chunk_id": self.chunk_id,
            "content_hash": self.content_hash,
            "source_path": self.source_path,
            "source_format": self.source_format,
            "element_type": self.element_type,
            "created_at": self.created_at.isoformat(),
        }
        if self.page_number is not None:
            md["page_number"] = self.page_number
        if self.slide_number is not None:
            md["slide_number"] = self.slide_number
        if self.sheet_name is not None:
            md["sheet_name"] = self.sheet_name
        if self.section_heading is not None:
            md["section_heading"] = self.section_heading
        if self.section_title is not None:
            md["section_title"] = self.section_title
        if self.row_range is not None:
            md["row_range"] = f"{self.row_range[0]}-{self.row_range[1]}"
        return md

In [11]:
# smoke test
if __name__ == "__main__":
    _t = RagChunk(
        text="Section 3.2: All employees must complete annual compliance training.",
        source_path="data/raw/test.pdf",
        source_format="pdf",
        page_number=1,
        element_type="text",
    )
    log.info(f"Schema OK | id: {_t.chunk_id} | hash: {_t.content_hash}")
    log.info(f"Metadata Sample: {_t.to_langchain_metadata()}")

2026-06-02 00:42:43,205 - INFO    | rag | Schema OK | id: c5ceff74-290d-4a1e-b7c3-76e7f8eba905 | hash: 962912e80c914a92
2026-06-02 00:42:43,205 - INFO    | rag | Metadata Sample: {'chunk_id': 'c5ceff74-290d-4a1e-b7c3-76e7f8eba905', 'content_hash': '962912e80c914a92', 'source_path': 'data/raw/test.pdf', 'source_format': 'pdf', 'element_type': 'text', 'created_at': '2026-06-01T16:42:43.205200+00:00', 'page_number': 1}


In [12]:
## docling parser
from docling.document_converter import DocumentConverter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
class DoclingParser:
    """Phase 0 parser for PDF/PPTX/DOCX/HTML/MD via IBM Docling.

    Pipeline: file → Docling DocumentConverter → markdown → recursive split → RagChunk[].
    Provenance at this stage is file-level (source_path). Page/slide-level
    provenance is added in Phase 1 with HybridChunker.
    """

    SUPPORTED: dict[str, SourceFormat] = {
    ".pdf": "pdf",
    ".pptx": "pptx",
    ".docx": "docx",
    ".html": "html",
    ".htm": "html",
    ".md": "md",
    }

    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 120):
        self.converter = DocumentConverter()
        # Separators ordered from "strong semantic boundary" → "last resort".
        # Markdown headings come first so chunks rarely cross sections.
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", ". ", " ", ""],
        )

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"DoclingParser does not support {path.suffix}")

        source_format = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[Docling] parsing {path.name} ({source_format}) …")

        try:
            result = self.converter.convert(str(path))
        except Exception as e:
            log.error(f"[Docling] convert failed for {path.name}: {e}")
            return []

        markdown = result.document.export_to_markdown()
        if not markdown.strip():
            log.warning(f"[Docling] {path.name}: empty output")
            return []

        rel_source = self._relative_source(path)
        texts = self.splitter.split_text(markdown)

        chunks = [
            RagChunk(
                text=t,
                source_path=rel_source,
                source_format=source_format,
                element_type="text",
            )
            for t in texts if t.strip()
        ]
        log.info(f"[Docling] {path.name}: {len(chunks)} chunks")
        return chunks

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [14]:
Path

pathlib.Path

In [84]:
parser = DoclingParser(chunk_size=cfg.CHUNK_SIZE, chunk_overlap=cfg.CHUNK_OVERLAP)

# Find the first PDF/PPTX/DOCX in data/raw/
candidates: list[Path] = []
for ext in (".pdf", ".pptx", ".docx", ".md", ".html", ".htm", ):
    candidates.extend(cfg.DATA_RAW_DIR.rglob(f"*{ext}"))

if not candidates:
    log.warning("No PDF/PPTX/DOCX found under data/raw/ — drop one in and re-run")
else:
    sample = candidates[0]
    sample_chunks = parser.parse(sample)
    print(f"\nFile           : {sample.name}")
    print(f"Total chunks   : {len(sample_chunks)}")
    if sample_chunks:
        c = sample_chunks[0]
        print(f"\n--- First chunk ---")
        print(f"format         : {c.source_format}")
        print(f"length (chars) : {len(c.text)}")
        print(f"chunk_id       : {c.chunk_id[:8]}…")
        print(f"hash           : {c.content_hash}")
        print(f"\ntext preview:\n{c.text[:400]}")
        print(f"\nmetadata sample:\n{c.to_langchain_metadata()}")

2026-06-01 12:27:38,016 - INFO    | rag | [Docling] parsing IPC_OLD_split_2.pdf (pdf) …
2026-06-01 12:27:38,074 - INFO    | docling.document_converter | Going to convert document batch...
2026-06-01 12:27:38,663 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cpu'
2026-06-01 12:27:39,801 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cpu'
/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
2026-06-01 12:27:43,935 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cpu'
2026-06-01 12:27:44,116 - INFO    | docling.pipeline.base_pipeline | Processing document IPC_OLD_split_2.pdf
2026-06-01 12:27:47,773 - INFO    | docling.document_converter | Finished converting document IPC_OLD_split_2.pdf in 9.76 sec.
2026-06-01 12:27:47,782 - INFO    | rag | [Docling] IPC_OLD_split_2.pdf: 15 chunks



File           : IPC_OLD_split_2.pdf
Total chunks   : 15

--- First chunk ---
format         : pdf
length (chars) : 620
chunk_id       : d4a8f1cb…
hash           : c9ce8ac455e8ef64

text preview:
## Illustration

A writes  his name  on the  back of  a bill  of exchange.  As the of this endorsement is to transfer the right to the bill to any who  may become  the lawful  holder of it, the endorsement is a security". effect person "valuable

31.

will". "A

31.  "A  will".--The  words  "a  will"  denote  any  testamentary document.

32.

referring to acts include illegal omissions. Words

32.

metadata sample:
{'chunk_id': 'd4a8f1cb-4ece-49af-9ffc-74b4e3130ee1', 'content_hash': 'c9ce8ac455e8ef64', 'source_path': '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_2.pdf', 'source_format': 'pdf', 'element_type': 'text', 'created_at': '2026-06-01T04:27:47.781823+00:00'}


In [15]:
## structured data parser
import json
import pandas as pd


class StructuredDataParser:
    """Phase 0 parser for CSV/XLSX/JSON/JSONL/TXT.

    Tabular: row → chunk (configurable batch), context prepended.
    JSON: list-of-records → record-per-chunk; else pretty-printed + split.
    JSONL: line → record → chunk.
    TXT: recursive split.
    """

    SUPPORTED: dict[str, SourceFormat] = {
        ".csv": "csv",
        ".xlsx": "xlsx",
        ".xls": "xlsx",
        ".json": "json",
        ".jsonl": "jsonl",
        ".ndjson": "jsonl",
        ".txt": "txt",
    }

    def __init__(
        self,
        rows_per_chunk: int = 1,
        max_rows_warn: int = 10_000,
        chunk_size: int = 800,
        chunk_overlap: int = 120,
    ):
        self.rows_per_chunk = rows_per_chunk
        self.max_rows_warn = max_rows_warn
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"StructuredDataParser does not support {path.suffix}")

        fmt = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[Structured] parsing {path.name} ({fmt}) …")
        try:
            dispatch = {
                "csv": self._parse_csv,
                "xlsx": self._parse_xlsx,
                "json": self._parse_json,
                "jsonl": self._parse_jsonl,
                "txt": self._parse_txt,
            }
            return dispatch[fmt](path)
        except Exception as e:
            log.error(f"[Structured] failed on {path.name}: {e}")
            return []

    # ---------- CSV / XLSX ----------
    def _parse_csv(self, path: Path) -> list[RagChunk]:
        df = pd.read_csv(path)
        if len(df) > self.max_rows_warn:
            log.warning(f"{path.name}: {len(df)} rows — consider increasing rows_per_chunk")
        return self._rows_to_chunks(df, path, "csv", sheet_name=None)

    def _parse_xlsx(self, path: Path) -> list[RagChunk]:
        chunks: list[RagChunk] = []
        xl = pd.ExcelFile(path)
        for sheet in xl.sheet_names:
            df = xl.parse(sheet)
            if df.empty:
                continue
            chunks.extend(self._rows_to_chunks(df, path, "xlsx", sheet_name=sheet))
        return chunks

    def _rows_to_chunks(
        self,
        df: pd.DataFrame,
        path: Path,
        source_format: SourceFormat,
        sheet_name: str | None,
    ) -> list[RagChunk]:
        rel = self._relative_source(path)
        n = len(df)
        if n == 0:
            return []

        scope = f"File: {path.name}"
        if sheet_name:
            scope += f" | Sheet: {sheet_name}"

        chunks: list[RagChunk] = []
        for start in range(0, n, self.rows_per_chunk):
            end = min(start + self.rows_per_chunk, n)
            sub = df.iloc[start:end]
            row_blocks = [
                "\n".join(f"{col}: {self._format_value(val)}" for col, val in row.items())
                for _, row in sub.iterrows()
            ]
            text = f"{scope} | Rows: {start+1}-{end}\n\n" + "\n\n---\n\n".join(row_blocks)
            chunks.append(RagChunk(
                text=text,
                source_path=rel,
                source_format=source_format,
                sheet_name=sheet_name,
                row_range=(start + 1, end),
                element_type="table",
            ))
        return chunks

    # ---------- JSON / JSONL ----------
    def _parse_json(self, path: Path) -> list[RagChunk]:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        rel = self._relative_source(path)

        # List of dicts → record-per-chunk
        if isinstance(data, list) and data and all(isinstance(x, dict) for x in data):
            return [
                self._record_to_chunk(rec, idx, path, rel, "json")
                for idx, rec in enumerate(data)
            ]

        # Dict with single dominant list field → use that list
        if isinstance(data, dict):
            list_keys = [k for k, v in data.items() if isinstance(v, list) and len(v) > 1]
            if len(list_keys) == 1 and all(isinstance(x, dict) for x in data[list_keys[0]]):
                key = list_keys[0]
                return [
                    self._record_to_chunk(rec, idx, path, rel, "json", parent_key=key)
                    for idx, rec in enumerate(data[key])
                ]

        # Fallback: pretty-print whole document, split if large
        text = f"File: {path.name}\n\n{json.dumps(data, indent=2, ensure_ascii=False)}"
        parts = self.splitter.split_text(text) if len(text) > 2000 else [text]
        return [
            RagChunk(text=p, source_path=rel, source_format="json", element_type="text")
            for p in parts if p.strip()
        ]

    def _parse_jsonl(self, path: Path) -> list[RagChunk]:
        rel = self._relative_source(path)
        chunks: list[RagChunk] = []
        with path.open("r", encoding="utf-8") as f:
            for idx, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    log.warning(f"{path.name}: line {idx+1} not valid JSON, skipping")
                    continue
                chunks.append(self._record_to_chunk(rec, idx, path, rel, "jsonl"))
        if len(chunks) > self.max_rows_warn:
            log.warning(f"{path.name}: {len(chunks)} chunks produced")
        return chunks

    def _record_to_chunk(
        self,
        rec: Any,
        idx: int,
        path: Path,
        rel: str,
        fmt: SourceFormat,
        parent_key: str | None = None,
    ) -> RagChunk:
        if isinstance(rec, dict):
            body = "\n".join(f"{k}: {self._format_value(v)}" for k, v in rec.items())
        else:
            body = json.dumps(rec, ensure_ascii=False)
        scope = f"File: {path.name}"
        if parent_key:
            scope += f" | Field: {parent_key}"
        scope += f" | Record: {idx+1}"
        return RagChunk(
            text=f"{scope}\n\n{body}",
            source_path=rel,
            source_format=fmt,
            row_range=(idx + 1, idx + 1),
            element_type="text",
        )

    # ---------- TXT ----------
    def _parse_txt(self, path: Path) -> list[RagChunk]:
        text = path.read_text(encoding="utf-8")
        if not text.strip():
            return []
        rel = self._relative_source(path)
        return [
            RagChunk(text=t, source_path=rel, source_format="txt", element_type="text")
            for t in self.splitter.split_text(text) if t.strip()
        ]

    # ---------- Helpers ----------
    @staticmethod
    def _format_value(val: Any) -> str:
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return ""
        if isinstance(val, (dict, list)):
            return json.dumps(val, ensure_ascii=False)
        return str(val)

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [86]:
struct_parser = StructuredDataParser(
    rows_per_chunk=1,
    chunk_size=cfg.CHUNK_SIZE,
    chunk_overlap=cfg.CHUNK_OVERLAP,
)

candidates: list[Path] = []
for ext in (".csv", ".xlsx", ".json", ".jsonl", ".txt", ".ndjson", ".yaml", ".yml", ".parquet", ".avro", ".orc", ".tsv", ".log", ):
    candidates.extend(cfg.DATA_RAW_DIR.rglob(f"*{ext}"))

if not candidates:
    log.warning("No CSV/XLSX/JSON/JSONL/TXT/NDJSON/YAML/YML/PARQUET/AVRO/ORC/TSV/LOG found under data/raw/ — drop a few in to test")
else:
    for sample in candidates[:]:  # preview up to 5 files
        result = struct_parser.parse(sample)
        print(f"\n=== {sample.relative_to(PROJECT_ROOT)} ===")
        print(f"  chunks   : {len(result)}")
        if result:
            c = result[0]
            print(f"  format   : {c.source_format} | element: {c.element_type}")
            print(f"  preview  : {c.text[:300]!r}")
            print(f"  metadata : {c.to_langchain_metadata()}")

2026-06-01 12:27:55,092 - WARNING | rag | No CSV/XLSX/JSON/JSONL/TXT/NDJSON/YAML/YML/PARQUET/AVRO/ORC/TSV/LOG found under data/raw/ — drop a few in to test


In [16]:
# [█████░░░░░░░░░░░░░░░░░░░░░░░░░░░░] ~15%

# PHASE 0 — Naive RAG ◄── WE ARE HERE
#    ✓ Config + schema
#    ✓ Parsers (Docling track + structured track)
#    ◯ Parser dispatcher          ← next (Piece 5)
#    ◯ Embedding + Chroma vector store
#    ◯ Top-k retrieval
#    ◯ Generation with Ollama (gemma-4-e4b)
#    ◯ End-to-end answer with citations

# PHASE 1 — Hybrid Retrieval
# PHASE 2 — Query Intelligence
# PHASE 3 — Agentic Orchestration (LangGraph)
# PHASE 4 — Evaluation & Observability
# PHASE 5 — Production Hardening (FastAPI service)
# PHASE 6 — Cloud & Scale (Docker, Azure/AWS/GCP)

In [17]:
from collections import defaultdict
from itertools import groupby
from tqdm.auto import tqdm


class ParserDispatcher:
    """Routes files to the right parser; aggregates RagChunks across a corpus.

    Extension point: append new parsers to `parsers` — first match wins.
    """

    def __init__(self, parsers: list):
        if not parsers:
            raise ValueError("At least one parser required")
        self.parsers = parsers

    def parse_file(self, path: Path) -> list[RagChunk]:
        for parser in self.parsers:
            if parser.supports(path):
                return parser.parse(path)
        return []

    def parse_directory(
        self,
        directory: Path,
        recursive: bool = True,
    ) -> list[RagChunk]:
        if not directory.exists():
            raise FileNotFoundError(f"Directory not found: {directory}")

        files = self._discover_files(directory, recursive)
        if not files:
            log.warning(f"No supported files under {directory}")
            return []

        log.info(f"Found {len(files)} supported file(s) under {directory.name}/")
        all_chunks: list[RagChunk] = []
        files_per_fmt: dict[str, int] = defaultdict(int)
        chunks_per_fmt: dict[str, int] = defaultdict(int)
        failed: list[str] = []
        seen_hashes: set[str] = set()
        dup_count = 0

        for fp in tqdm(files, desc="Parsing", unit="file"):
            try:
                chunks = self.parse_file(fp)
            except Exception as e:
                log.error(f"Parse error on {fp.name}: {e}")
                failed.append(fp.name)
                continue
            if not chunks:
                continue

            fmt = chunks[0].source_format
            files_per_fmt[fmt] += 1
            for chunk in chunks:
                if chunk.content_hash in seen_hashes:
                    dup_count += 1
                    continue
                seen_hashes.add(chunk.content_hash)
                all_chunks.append(chunk)
                chunks_per_fmt[fmt] += 1

        self._print_summary(files_per_fmt, chunks_per_fmt, failed, dup_count, len(all_chunks))
        return all_chunks

    def _discover_files(self, directory: Path, recursive: bool) -> list[Path]:
        pattern = "**/*" if recursive else "*"
        files: list[Path] = []
        for p in directory.glob(pattern):
            if not p.is_file():
                continue
            if any(part.startswith(".") for part in p.parts):
                continue
            if any(parser.supports(p) for parser in self.parsers):
                files.append(p)
        return sorted(files)

    @staticmethod
    def _print_summary(files_per_fmt, chunks_per_fmt, failed, dup_count, total) -> None:
        log.info("=" * 56)
        log.info(f"{'Format':<10} {'Files':>10} {'Chunks':>12}")
        log.info("-" * 56)
        for fmt in sorted(files_per_fmt):
            log.info(f"{fmt:<10} {files_per_fmt[fmt]:>10} {chunks_per_fmt[fmt]:>12}")
        log.info("-" * 56)
        log.info(f"Total unique chunks : {total}")
        log.info(f"Duplicates skipped  : {dup_count}")
        if failed:
            shown = failed[:5]
            extra = f" (+{len(failed)-5} more)" if len(failed) > 5 else ""
            log.info(f"Failed files        : {len(failed)} — {shown}{extra}")
        log.info("=" * 56)


# --- Cache helpers (so we don't re-parse on every notebook re-run) ---

def save_chunks_cache(chunks: list[RagChunk], path: Path) -> None:
    payload = [c.model_dump(mode="json") for c in chunks]
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    log.info(f"Cached {len(chunks)} chunks → {path.relative_to(PROJECT_ROOT)}")


def load_chunks_cache(path: Path) -> list[RagChunk]:
    if not path.exists():
        return []
    data = json.loads(path.read_text(encoding="utf-8"))
    chunks = [RagChunk(**item) for item in data]
    log.info(f"Loaded {len(chunks)} chunks ← {path.relative_to(PROJECT_ROOT)}")
    return chunks

In [18]:
dispatcher = ParserDispatcher(parsers=[
    DoclingParser(chunk_size=cfg.CHUNK_SIZE, chunk_overlap=cfg.CHUNK_OVERLAP),
    StructuredDataParser(
        rows_per_chunk=1,
        chunk_size=cfg.CHUNK_SIZE,
        chunk_overlap=cfg.CHUNK_OVERLAP,
    ),
])


In [19]:

CACHE_PATH = cfg.DATA_PROCESSED_DIR / "phase0_chunks.json"

# Force re-parse by setting REPARSE = True
REPARSE = False

if not REPARSE and CACHE_PATH.exists():
    all_chunks = load_chunks_cache(CACHE_PATH)
else:
    all_chunks = dispatcher.parse_directory(cfg.DATA_RAW_DIR)
    if all_chunks:
        save_chunks_cache(all_chunks, CACHE_PATH)

# Quick peek: one sample chunk per format
if all_chunks:
    print("\nSample chunk per format:")
    by_fmt = sorted(all_chunks, key=lambda c: c.source_format)
    for fmt, group in groupby(by_fmt, key=lambda c: c.source_format):
        sample = next(group)
        print(f"\n--- {fmt} ---")
        print(f"  source : {sample.source_path}")
        print(f"  chars  : {len(sample.text)}")
        print(f"  text   : {sample.text[:220]!r}")
else:
    log.warning("No chunks produced — make sure data/raw/ contains supported files")

2026-06-02 00:43:52,964 - INFO    | rag | Loaded 1677 chunks ← data/processed/phase0_chunks.json



Sample chunk per format:

--- pdf ---
  source : /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_0.pdf
  chars  : 108
  text   : 'INDIAN PENAL CODE, 1860 THE\n\nNO. 45 OF 1860 1* ACT\n\nOctober, 1860.] [6th\n\nI CHAPTER\n\nINTRODUCTION\n\nCHAPTER I'


In [20]:
from pathlib import Path
for p in dispatcher.parsers:
    print(f"{type(p).__name__}: supports .txt? {p.supports(Path('x.txt'))}")

DoclingParser: supports .txt? False
StructuredDataParser: supports .txt? True


In [21]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings


COLLECTION_NAME = "IPC_Corpus"

# 1) Embedding function — local, via Ollama
embeddings = OllamaEmbeddings(
    model=cfg.EMBEDDING_MODEL,
    base_url=cfg.OLLAMA_BASE_URL,
)

# Probe: confirm the model is reachable and capture dimension
_probe = embeddings.embed_query("hello world")
EMBED_DIM = len(_probe)
log.info(f"Embedding model '{cfg.EMBEDDING_MODEL}' OK → dim={EMBED_DIM}")


# 2) RagChunk → LangChain Document at the boundary
def chunks_to_documents(chunks: list[RagChunk]) -> tuple[list[Document], list[str]]:
    docs = [
        Document(page_content=c.text, metadata=c.to_langchain_metadata())
        for c in chunks
    ]
    ids = [c.chunk_id for c in chunks]
    return docs, ids



2026-06-02 00:44:20,665 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:44:20,667 - INFO    | rag | Embedding model 'embeddinggemma:latest' OK → dim=768


In [22]:

# 3) Open (or create) the persistent Chroma collection
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(cfg.CHROMA_PERSIST_DIR),
)

existing_count = vectorstore._collection.count()
log.info(f"Chroma '{COLLECTION_NAME}': {existing_count} vectors present")


# 4) Idempotent ingest — embed only NEW chunks
REINDEX = False  # flip to True to wipe and rebuild from scratch

if REINDEX and existing_count > 0:
    log.warning(f"REINDEX=True → wiping {existing_count} vectors")
    vectorstore.delete_collection()
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(cfg.CHROMA_PERSIST_DIR),
    )
    existing_count = 0

if not all_chunks:
    log.warning("`all_chunks` is empty — run Piece 5 first, or load from cache")
else:
    docs, ids = chunks_to_documents(all_chunks)

    # Skip chunks already indexed (by chunk_id)
    if existing_count > 0:
        existing_ids = set(vectorstore.get(include=[])["ids"])
        to_add = [(d, i) for d, i in zip(docs, ids) if i not in existing_ids]
    else:
        to_add = list(zip(docs, ids))

    log.info(
        f"To embed: {len(to_add)} new ("
        f"{len(docs) - len(to_add)} already present)"
    )

    # Batch for progress visibility — Ollama on CPU can be slow at first
    BATCH_SIZE = 64
    for i in tqdm(range(0, len(to_add), BATCH_SIZE), desc="Embedding", unit="batch"):
        batch = to_add[i : i + BATCH_SIZE]
        if not batch:
            continue
        b_docs = [p[0] for p in batch]
        b_ids = [p[1] for p in batch]
        vectorstore.add_documents(documents=b_docs, ids=b_ids)

    final_count = vectorstore._collection.count()
    log.info(f"Chroma '{COLLECTION_NAME}': {final_count} vectors total")

2026-06-02 00:47:30,381 - INFO    | chromadb.telemetry.product.posthog | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-06-02 00:47:31,425 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-06-02 00:47:31,427 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
2026-06-02 00:47:31,596 - INFO    | rag | Chroma 'IPC_Corpus': 613 vectors present
2026-06-02 00:47:31,604 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
2026-06-02 00:47:32,090 - INFO    | rag | To embed: 1677 new (0 already present)


Embedding:   0%|          | 0/27 [00:00<?, ?batch/s]

2026-06-02 00:47:46,104 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:48:00,842 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:48:15,049 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:48:30,089 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:48:47,606 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:49:11,099 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:49:34,150 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:50:25,172 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:50:48,911 - INFO    | httpx | HTTP Request: POST http://localhost:11434/ap

In [23]:
def search_and_show(query: str, k: int | None = None) -> list[tuple[Document, float]]:
    k = k or cfg.TOP_K
    results = vectorstore.similarity_search_with_score(query, k=k)

    print(f"\n🔎 Query: {query!r}")
    print(f"Top {k} (Chroma cosine distance — lower = closer):\n")

    for rank, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc_bits = []
        if md.get("page_number"):   loc_bits.append(f"p.{md['page_number']}")
        if md.get("slide_number"):  loc_bits.append(f"slide {md['slide_number']}")
        if md.get("sheet_name"):    loc_bits.append(f"sheet '{md['sheet_name']}'")
        if md.get("row_range"):     loc_bits.append(f"rows {md['row_range']}")
        if md.get("section_title"): loc_bits.append(f"§ {md['section_title']}")
        loc = "  |  ".join(loc_bits)

        print(f"#{rank}  score={score:.4f}  [{md['source_format']}]  {md['source_path']}")
        if loc:
            print(f"     {loc}")
        preview = doc.page_content[:300].replace("\n", " ")
        print(f"     {preview!r}\n")

    return results




In [24]:
# Change this to something that actually appears in YOUR corpus
QUERY = "Child labor laws"
_ = search_and_show(QUERY)

2026-06-02 00:59:44,222 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 00:59:44,224 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔎 Query: 'Child labor laws'
Top 5 (Chroma cosine distance — lower = closer):

#1  score=0.9391  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf
     '## Of right of private defence'

#2  score=1.1018  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_33.pdf
     '- (3) Where  any person,  not being the lawful guardian of a minor, or  uses such  minor for  the purposes of begging, it shall be unless  the  contrary  is  proved,  that  he  kidnapped  or obtained  the custody  of that minor in order that the minor be employed or used for the purposes of begging.'

#3  score=1.1302  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_11.pdf
     p.5  |  § Of offences against child
     'Of offences against child - 93. Exposure and abandonment of child under twelve years of age, by parent or person having care of it. -Whoever being the father or mother of a child under the age of twelve years, or having the care of such child, shall expose or leave such child in 

In [25]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(
    model=cfg.OLLAMA_MODEL,
    base_url=cfg.OLLAMA_BASE_URL,
    temperature=cfg.LLM_TEMPERATURE,
    num_ctx=cfg.LLM_NUM_CTX,
)

_smoke = llm.invoke("Reply with exactly: pong")
log.info(f"LLM '{cfg.OLLAMA_MODEL}' OK → {_smoke.content!r}")

2026-06-02 01:01:25,141 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-02 01:01:25,271 - INFO    | rag | LLM 'gemma-4-e4b:latest' OK → 'pong'


In [26]:
SYSTEM_PROMPT = """You are a precise assistant for company policy and internal documentation questions.

Rules:
1. Use ONLY the information in the CONTEXT below. Do not use outside knowledge.
2. If the answer is not in the context, reply exactly: "I don't have that information in the provided documents."
3. Cite every claim with the source tag in square brackets, e.g. [1] or [2, 3].
4. Quote short passages verbatim when they are decisive (e.g., policy clauses, IDs, dates).
5. Be concise. Do not pad or speculate."""


def _format_location(md: dict) -> str:
    bits = []
    if md.get("page_number"):   bits.append(f"p.{md['page_number']}")
    if md.get("slide_number"):  bits.append(f"slide {md['slide_number']}")
    if md.get("sheet_name"):    bits.append(f"sheet '{md['sheet_name']}'")
    if md.get("row_range"):     bits.append(f"rows {md['row_range']}")
    if md.get("section_title"): bits.append(f"§ {md['section_title']}")
    return " | ".join(bits)


def build_context(results: list[tuple[Document, float]]) -> tuple[str, list[dict]]:
    """Format retrieved chunks as numbered context + return a citation table."""
    blocks: list[str] = []
    citations: list[dict] = []
    for i, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc = _format_location(md)
        header = f"[{i}] {md['source_path']}" + (f" | {loc}" if loc else "")
        blocks.append(f"{header}\n{doc.page_content}")
        citations.append({
            "tag": i,
            "source_path": md["source_path"],
            "source_format": md["source_format"],
            "location": loc,
            "score": float(score),
            "chunk_id": md.get("chunk_id"),
        })
    return "\n\n---\n\n".join(blocks), citations


def answer(query: str, k: int | None = None, show_context: bool = False) -> dict:
    """End-to-end RAG: retrieve → build prompt → generate → return structured result."""
    k = k or cfg.TOP_K
    results = vectorstore.similarity_search_with_score(query, k=k)

    if not results:
        return {
            "question": query,
            "answer": "I don't have any relevant documents indexed.",
            "citations": [],
        }

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])

    out = {
        "question": query,
        "answer": response.content,
        "citations": citations,
    }
    if show_context:
        out["context"] = context_block
    return out


def pretty_print(result: dict) -> None:
    print(f"{result['question']}\n")
    print(f"{result['answer']}\n")
    if result["citations"]:
        print("Sources:")
        for c in result["citations"]:
            loc = f"  |  {c['location']}" if c["location"] else ""
            print(f"   [{c['tag']}] {c['source_path']}{loc}   (dist={c['score']:.3f})")

In [102]:
# Try a query that should be answerable from YOUR corpus
QUERY = "sexual assault"

result = answer(QUERY)
pretty_print(result)

# Sanity check the model honours "I don't know"
print("\n" + "=" * 60 + "\n")
nonsense = answer("what's the recipe for chocolate cake")
pretty_print(nonsense)

2026-06-01 12:43:50,085 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:43:55,211 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 12:44:33,061 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


sexual assault

I don't have that information in the provided documents.

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_11.pdf   (dist=0.733)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_21.pdf   (dist=0.891)
   [3] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf   (dist=0.897)
   [4] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf   (dist=0.956)
   [5] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_13.pdf   (dist=0.971)




2026-06-01 12:44:37,026 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


what's the recipe for chocolate cake

I don't have that information in the provided documents.

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf   (dist=1.197)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_18.pdf   (dist=1.245)
   [3] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_12.pdf   (dist=1.286)
   [4] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_12.pdf  |  p.1   (dist=1.286)
   [5] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_19.pdf   (dist=1.287)


In [103]:
# [Files in data/raw/]
#         ↓  ParserDispatcher
#    [RagChunk[]]
#         ↓  OllamaEmbeddings (embeddinggemma)
#    [Chroma index]
#         ↓  similarity_search_with_score
#    [Top-k chunks]
#         ↓  build_context → SYSTEM_PROMPT
#    [Gemma (gemma-4-e4b)]
#         ↓
#    [Answer + citations]

In [104]:
# [████████████░░░░░░░░░░░░░░░░░░░░░░] ~35%

# ✓ PHASE 0 — Naive RAG     (DONE — your baseline)
#   → PHASE 1 — Hybrid Retrieval ◄── NEXT
#     PHASE 2 — Query Intelligence
#     PHASE 3 — Agentic Orchestration
#     PHASE 4 — Evaluation & Observability
#     PHASE 5 — Production Hardening
#     PHASE 6 — Cloud & Scale

In [105]:
# phase 1
# hybrid retrieval

In [27]:
import re
from rank_bm25 import BM25Okapi


class BM25Retriever:
    """In-memory BM25 sparse retriever over RagChunks.

    Returns LangChain Documents so it composes with the dense retriever
    under the same interface. Score is BM25 (higher = better) — opposite
    of Chroma distance (lower = better). RRF fusion in the next piece
    sidesteps this by using rank, not raw score.
    """

    # Keep hyphens (vendor IDs like V-001), drop other punctuation.
    _TOKEN_RE = re.compile(r"[^\w\s\-]")

    def __init__(self, chunks: list[RagChunk], min_token_len: int = 2):
        if not chunks:
            raise ValueError("BM25Retriever needs at least one chunk")
        self.chunks = chunks
        self.min_token_len = min_token_len

        log.info(f"[BM25] tokenizing {len(chunks)} chunks …")
        self._tokenized_corpus = [self._tokenize(c.text) for c in chunks]
        self.bm25 = BM25Okapi(self._tokenized_corpus)

        # Pre-build LangChain Documents so retrieval is just a lookup
        self._docs = [
            Document(page_content=c.text, metadata=c.to_langchain_metadata())
            for c in chunks
        ]
        avg_tokens = sum(len(t) for t in self._tokenized_corpus) / len(chunks)
        log.info(f"[BM25] index ready | {len(chunks)} docs | avg {avg_tokens:.0f} tokens/doc")

    def _tokenize(self, text: str) -> list[str]:
        text = text.lower()
        text = self._TOKEN_RE.sub(" ", text)
        return [t for t in text.split() if len(t) >= self.min_token_len]

    def retrieve(
        self,
        query: str,
        k: int = 10,
        min_score: float = 0.0,
    ) -> list[tuple[Document, float]]:
        tokens = self._tokenize(query)
        if not tokens:
            return []
        scores = self.bm25.get_scores(tokens)
        # Argsort top-k
        ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return [
            (self._docs[i], float(scores[i]))
            for i in ranked_idx
            if scores[i] > min_score
        ]



In [28]:

# Build the index from chunks already in memory (or load from cache first if needed)
if not all_chunks:
    all_chunks = load_chunks_cache(cfg.DATA_PROCESSED_DIR / "phase0_chunks.json")

bm25_retriever = BM25Retriever(all_chunks)

2026-06-02 01:01:25,853 - INFO    | rag | [BM25] tokenizing 1677 chunks …
2026-06-02 01:01:25,901 - INFO    | rag | [BM25] index ready | 1677 docs | avg 82 tokens/doc


In [108]:
def show_results(title: str, results: list[tuple[Document, float]], score_label: str) -> None:
    print(f"\n--- {title} ---")
    if not results:
        print("  (no results)")
        return
    for rank, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc = _format_location(md)
        loc_str = f"  |  {loc}" if loc else ""
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"  #{rank}  {score_label}={score:.3f}  {md['source_path']}{loc_str}")
        print(f"        {preview!r}")



In [109]:

# Try BOTH a literal/keyword query and a semantic/paraphrase query
TEST_QUERIES = [
    "Federal agencies are in possession of documents pertaining to gross human rights violations abroad which are needed by foreign authorities to document",                              # literal ID — BM25 should win
    "Human Rights Information Act",        # exact-ish phrase — both should hit
    "Girls Count Act of 2014",   # paraphrase — dense should win
]

for q in TEST_QUERIES:
    print("=" * 70)
    print(f"🔎 Query: {q!r}")

    dense_results = vectorstore.similarity_search_with_score(q, k=5)
    bm25_results = bm25_retriever.retrieve(q, k=5)

    show_results("DENSE (Chroma, embeddinggemma)", dense_results, "dist")
    show_results("BM25  (rank_bm25)",              bm25_results,  "bm25")

2026-06-01 12:45:08,185 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


🔎 Query: 'Federal agencies are in possession of documents pertaining to gross human rights violations abroad which are needed by foreign authorities to document'

--- DENSE (Chroma, embeddinggemma) ---
  #1  dist=1.181  /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_45.pdf
        '. documents material.--Whoever material, any such appearance to possession device *[imprisonment for to'
  #2  dist=1.252  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf
        '## Of fraudulent deeds and dispositions of property'
  #3  dist=1.264  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf
        'Of wrongful restraint and wrongful confinement'
  #4  dist=1.289  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf
        '## Of receiving stolen property'
  #5  dist=1.310  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_4.pdf
        '## Of criminal misappropriation of property'

--- BM25  (rank_bm25) ---
  #1  bm25=23.482  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_4.

2026-06-01 12:45:08,300 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:45:08,398 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"



--- DENSE (Chroma, embeddinggemma) ---
  #1  dist=0.849  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf
        '## Of right of private defence'
  #2  dist=1.041  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf
        'Of wrongful restraint and wrongful confinement'
  #3  dist=1.141  /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_13.pdf
        'all  lawful means in his or their to prevent it and, in the event of its  taking place, do not use lawful  means  in  hi'
  #4  dist=1.143  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf
        '## Of fraudulent deeds and dispositions of property'
  #5  dist=1.144  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_15.pdf
        '## OF OFFENCES AGAINST THE PUBLIC TRANQUILLITY'

--- BM25  (rank_bm25) ---
  #1  bm25=7.380  /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_12.pdf
        '- (b) asserts, counsels, advises, propagates or publishes that any class of   persons   by  reason of their being member'
  #2

In [110]:
## combining both retrievers: 
## RRF_score (doc) = Σ over retrievers:  weight / (k + rank_of_doc_in_that_retriever)

In [29]:
def reciprocal_rank_fusion(
    rankings: list[list[Document]],
    k: int = 60,
    weights: list[float] | None = None,
) -> list[tuple[Document, float]]:
    """Fuse multiple ranked Document lists via Reciprocal Rank Fusion.

    Uses rank position only — immune to score-scale mismatches between
    retrievers. Documents are deduped by chunk_id from their metadata.
    """
    if weights is None:
        weights = [1.0] * len(rankings)
    if len(weights) != len(rankings):
        raise ValueError("weights must match number of rankings")

    scores: dict[str, float] = defaultdict(float)
    doc_lookup: dict[str, Document] = {}

    for ranking, w in zip(rankings, weights):
        for rank, doc in enumerate(ranking, start=1):
            doc_id = doc.metadata.get("chunk_id") or doc.metadata.get("content_hash")
            if not doc_id:
                continue
            scores[doc_id] += w / (k + rank)
            doc_lookup.setdefault(doc_id, doc)

    return sorted(
        ((doc_lookup[did], s) for did, s in scores.items()),
        key=lambda x: x[1],
        reverse=True,
    )


class EnsembleRetriever:
    """Hybrid retriever = dense ⊕ BM25, fused via RRF.

    fetch_k: how many to pull from EACH base retriever before fusion, default 20.
    top_k:   how many to return after fusion, default 5.
    rrf_k:   the "k" parameter for RRF fusion, default 60 (per the research paper's recommendation).

    Weights are per-retriever. Equal weights (1.0, 1.0) is a strong default.
    Bump dense weight up if your corpus is paraphrase-heavy; bump BM25 up
    if it's full of IDs / codes / exact references.
    """

    def __init__(
        self,
        dense_store: Chroma,
        sparse_retriever: BM25Retriever,
        fetch_k: int = 20, 
        weights: tuple[float, float] = (1.0, 1.0),
        rrf_k: int = 60,
    ):
        self.dense_store = dense_store
        self.sparse_retriever = sparse_retriever
        self.fetch_k = fetch_k
        self.weights = list(weights)
        self.rrf_k = rrf_k

    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Document, float]]:
        # Dense: we only need order, not the distance, for RRF
        dense_docs = self.dense_store.similarity_search(query, k=self.fetch_k)

        # Sparse: returns (doc, bm25_score); keep just docs, in rank order
        sparse_pairs = self.sparse_retriever.retrieve(query, k=self.fetch_k)
        sparse_docs = [d for d, _ in sparse_pairs]

        fused = reciprocal_rank_fusion(
            rankings=[dense_docs, sparse_docs],
            k=self.rrf_k,
            weights=self.weights,
        )
        return fused[:top_k]




In [30]:
ensemble = EnsembleRetriever(
    dense_store=vectorstore,
    sparse_retriever=bm25_retriever,
    fetch_k=20,
    weights=(1.0, 1.0),
)
log.info(f"[Ensemble] ready | fetch_k={ensemble.fetch_k} | weights={ensemble.weights}")

2026-06-02 01:09:30,177 - INFO    | rag | [Ensemble] ready | fetch_k=20 | weights=[1.0, 1.0]


In [31]:
## comparing dense vs. bm25 vs hybrid

In [32]:
# Redefine pretty_print to accept a score-label (replaces the Phase 0 version)
def pretty_print(result: dict, score_label: str = "score") -> None:
    print(f"{result['question']}\n")
    print(f"{result['answer']}\n")
    if result["citations"]:
        print("Sources:")
        for c in result["citations"]:
            loc = f"  |  {c['location']}" if c["location"] else ""
            print(f"   [{c['tag']}] {c['source_path']}{loc}   ({score_label}={c['score']:.3f})")


def answer_phase1(
    query: str,
    k: int | None = None,
    retriever: str = "hybrid",  # "dense" | "bm25" | "hybrid"
) -> dict:
    k = k or cfg.TOP_K

    if retriever == "dense":
        results = vectorstore.similarity_search_with_score(query, k=k)
    elif retriever == "bm25":
        results = bm25_retriever.retrieve(query, k=k)
    elif retriever == "hybrid":
        results = ensemble.retrieve(query, top_k=k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if not results:
        return {"question": query, "answer": "No relevant docs.", "citations": [], "retriever": retriever}

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])
    return {
        "question": query,
        "answer": response.content,
        "citations": citations,
        "retriever": retriever,
    }




In [ ]:
SCORE_LABELS = {"dense": "dist", "bm25": "bm25", "hybrid": "rrf"}

# Pick ONE query from your corpus and watch how the three retrievers differ
TEST_QUERY = "Right of private defence against the act of a person of unsound mind"

for r in ("dense", "bm25", "hybrid"):
    print("=" * 72)
    print(f"RETRIEVER: {r.upper()}")
    print("=" * 72)
    pretty_print(answer_phase1(TEST_QUERY, retriever=r), score_label=SCORE_LABELS[r])
    print()

In [115]:
## reranker: pulling from top 20 retrieved chuncks

In [33]:
from FlagEmbedding import FlagReranker


class Reranker:
    """Thin wrapper around BGE cross-encoder rerankers."""

    def __init__(
        self,
        model_name: str = "BAAI/bge-reranker-base",
        use_fp16: bool = True,
        normalize: bool = True,
    ):
        log.info(f"[Reranker] loading {model_name} (first call downloads ~1GB)…")
        self.model = FlagReranker(model_name, use_fp16=use_fp16)
        self.model_name = model_name
        self.normalize = normalize  # sigmoid → 0..1 scores, intuitive thresholds
        log.info(f"[Reranker] ready")

    def score(self, query: str, docs: list[Document]) -> list[float]:
        if not docs:
            return []
        pairs = [[query, d.page_content] for d in docs]
        out = self.model.compute_score(pairs, normalize=self.normalize)
        # FlagReranker returns float for a single pair, list for multiple
        if isinstance(out, float):
            return [out]
        return [float(x) for x in out]


class RerankedRetriever:
    """Wrap any base retriever; over-fetch then rerank with a cross-encoder.

    Pipeline:  base.retrieve(top_k=fetch_k)  →  rerank  →  top_k

    Cuts the noise that hybrid retrieval inevitably brings in.
    """

    def __init__(
        self,
        base_retriever,
        reranker: Reranker,
        fetch_k: int = 20,
        min_score: float | None = None,  # set in Phase 4 eval, not by vibes
    ):
        self.base = base_retriever
        self.reranker = reranker
        self.fetch_k = fetch_k
        self.min_score = min_score

    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Document, float]]:
        candidates = self.base.retrieve(query, top_k=self.fetch_k)
        if not candidates:
            return []

        docs = [d for d, _ in candidates]
        scores = self.reranker.score(query, docs)

        scored = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
        if self.min_score is not None:
            scored = [(d, s) for d, s in scored if s >= self.min_score]
        return scored[:top_k]


In [34]:
# download the reranker model
reranker = Reranker(model_name="BAAI/bge-reranker-base", use_fp16=True, normalize=True)

hybrid_reranked = RerankedRetriever(
    base_retriever=ensemble,    # our hybrid retriever from Cell 19
    reranker=reranker,
    fetch_k=20,
    min_score=None,             # learn this from eval, not intuition
)
log.info(f"[Pipeline] hybrid+rerank ready | fetch_k={hybrid_reranked.fetch_k}")

2026-06-02 01:09:45,528 - INFO    | rag | [Reranker] loading BAAI/bge-reranker-base (first call downloads ~1GB)…
/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
2026-06-02 01:09:51,549 - INFO    | rag | [Reranker] ready
2026-06-02 01:09:51,550 - INFO    | rag | [Pipeline] hybrid+rerank ready | fetch_k=20


In [35]:
## testing hybrid vs hybrid+rerank on the same query

In [36]:
def answer_phase1(
    query: str,
    k: int | None = None,
    retriever: str = "hybrid_reranked",   # new default
) -> dict:
    k = k or cfg.TOP_K

    if retriever == "dense":
        results = vectorstore.similarity_search_with_score(query, k=k)
    elif retriever == "bm25":
        results = bm25_retriever.retrieve(query, k=k)
    elif retriever == "hybrid":
        results = ensemble.retrieve(query, top_k=k)
    elif retriever == "hybrid_reranked":
        results = hybrid_reranked.retrieve(query, top_k=k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if not results:
        return {"question": query, "answer": "No relevant docs.", "citations": [], "retriever": retriever}

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])
    return {
        "question": query,
        "answer": response.content,
        "citations": citations,
        "retriever": retriever,
    }



In [ ]:

SCORE_LABELS = {
    "dense": "dist", "bm25": "bm25", "hybrid": "rrf", "hybrid_reranked": "rerank",
}

# Use a query where the right answer needs precision, not just recall
TEST_QUERY = "Right of private defence"

for r in ("hybrid", "hybrid_reranked"):
    print("=" * 72)
    print(f"RETRIEVER: {r.upper()}")
    print("=" * 72)
    pretty_print(answer_phase1(TEST_QUERY, retriever=r), score_label=SCORE_LABELS[r])
    print()

In [37]:
#pip install "docling-core[chunking]" --break-system-packages

## phase 1: hybrid chunker

In [38]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer


class DoclingHybridParser:
    """Phase 1: structure-aware chunking with real provenance.

    vs DoclingParser (Phase 0):
      • chunks respect headings/tables/sections (no mid-table slicing)
      • real page_number (PDF) / slide_number (PPTX)
      • section hierarchy captured in section_title
      • heading context prepended to text → better embeddings
    """

    SUPPORTED: dict[str, SourceFormat] = {
        ".pdf": "pdf", ".pptx": "pptx", ".docx": "docx",
        ".html": "html", ".htm": "html", ".md": "md",
    }

    def __init__(
        self,
        max_tokens: int = 512,
        tokenizer_id: str = "sentence-transformers/all-MiniLM-L6-v2",
        merge_peers: bool = True,
    ):
        self.converter = DocumentConverter()
        tok = HuggingFaceTokenizer(
            tokenizer=AutoTokenizer.from_pretrained(tokenizer_id),
            max_tokens=max_tokens,
        )
        self.chunker = HybridChunker(tokenizer=tok, merge_peers=merge_peers)

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"DoclingHybridParser does not support {path.suffix}")

        fmt = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[DoclingHybrid] parsing {path.name} ({fmt}) …")
        try:
            result = self.converter.convert(str(path))
        except Exception as e:
            log.error(f"[DoclingHybrid] convert failed for {path.name}: {e}")
            return []

        doc = result.document
        rel = self._relative_source(path)
        chunks: list[RagChunk] = []

        for dl_chunk in self.chunker.chunk(dl_doc=doc):
            # contextualize() prepends the heading hierarchy → richer embeddings
            text = self.chunker.contextualize(chunk=dl_chunk)
            if not text or not text.strip():
                continue

            page_no = self._first_page_no(dl_chunk)
            chunks.append(RagChunk(
                text=text,
                source_path=rel,
                source_format=fmt,
                page_number=page_no if fmt == "pdf" else None,
                slide_number=page_no if fmt == "pptx" else None,
                section_title=self._section_title(dl_chunk),
                element_type=self._element_type(dl_chunk),
            ))

        log.info(f"[DoclingHybrid] {path.name}: {len(chunks)} structure-aware chunks")
        return chunks

    @staticmethod
    def _first_page_no(dl_chunk) -> int | None:
        try:
            for item in dl_chunk.meta.doc_items:
                for prov in getattr(item, "prov", []) or []:
                    pn = getattr(prov, "page_no", None)
                    if pn is not None:
                        return int(pn)
        except Exception:
            pass
        return None

    @staticmethod
    def _section_title(dl_chunk) -> str | None:
        try:
            headings = getattr(dl_chunk.meta, "headings", None)
            if headings:
                return " > ".join(h for h in headings if h)[:300]
        except Exception:
            pass
        return None

    @staticmethod
    def _element_type(dl_chunk) -> ElementType:
        try:
            for item in dl_chunk.meta.doc_items:
                label = str(getattr(item, "label", "")).lower()
                if "table" in label: return "table"
                if "list" in label:  return "list"
                if "code" in label:  return "code"
        except Exception:
            pass
        return "text"

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [ ]:

SCORE_LABELS = {
    "dense": "dist", "bm25": "bm25", "hybrid": "rrf", "hybrid_reranked": "rerank",
}

# Use a query where the right answer needs precision, not just recall
TEST_QUERY = "Right of private defence"

for r in ("hybrid", "hybrid_reranked"):
    print("=" * 72)
    print(f"RETRIEVER: {r.upper()}")
    print("=" * 72)
    pretty_print(answer_phase1(TEST_QUERY, retriever=r), score_label=SCORE_LABELS[r])
    print()

In [39]:
## re-parse + cheap peek
dispatcher_v2 = ParserDispatcher(parsers=[
    DoclingHybridParser(max_tokens=512, merge_peers=True),
    StructuredDataParser(
        rows_per_chunk=1,
        chunk_size=cfg.CHUNK_SIZE,
        chunk_overlap=cfg.CHUNK_OVERLAP,
    ),
])

all_chunks_v2 = dispatcher_v2.parse_directory(cfg.DATA_RAW_DIR)
save_chunks_cache(all_chunks_v2, cfg.DATA_PROCESSED_DIR / "phase1_chunks.json")

# Confirm page_number + section_title are now populated on a PDF chunk
pdf_chunks = [c for c in all_chunks_v2 if c.source_format == "pdf"]
if pdf_chunks:
    sample = next((c for c in pdf_chunks if c.page_number is not None), pdf_chunks[0])
    print("Structure-aware PDF chunk:")
    print(f"  page_number   : {sample.page_number}")
    print(f"  section_title : {sample.section_title}")
    print(f"  element_type  : {sample.element_type}")
    print(f"  text preview  : {sample.text[:320]!r}")
else:
    log.warning("No PDF chunks — add a PDF to data/raw/pdfs/ to see page numbers")

/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2026-06-02 01:10:27,122 - INFO    | rag | Found 74 supported file(s) under IPC/


Parsing:   0%|          | 0/74 [00:00<?, ?file/s]

2026-06-02 01:10:27,127 - INFO    | rag | [DoclingHybrid] parsing IPC_OLD_split_0.pdf (pdf) …
2026-06-02 01:10:27,366 - INFO    | docling.document_converter | Going to convert document batch...
2026-06-02 01:10:28,215 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cpu'
2026-06-02 01:10:30,527 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cpu'
/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
2026-06-02 01:10:37,609 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cpu'
2026-06-02 01:10:41,127 - INFO    | docling.pipeline.base_pipeline | Processing document IPC_OLD_split_0.pdf
2026-06-02 01:10:45,657 - INFO    | docling.document_converter | Finished converting document IPC_OLD_split_0.pdf in 18.53 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (60

Structure-aware PDF chunk:
  page_number   : 1
  section_title : None
  element_type  : text
  text preview  : 'INDIAN PENAL CODE, 1860 THE\nNO. 45 OF 1860 1* ACT\nOctober, 1860.] [6th\nI CHAPTER\nINTRODUCTION\nCHAPTER I'


In [122]:
all_chunks = all_chunks_v2  # promote to the working corpus

# 1) Chroma: chunk boundaries changed → full re-embed required
log.warning("Rebuilding Chroma with structure-aware chunks (one-time re-embed)…")
vectorstore.delete_collection()
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(cfg.CHROMA_PERSIST_DIR),
)
docs, ids = chunks_to_documents(all_chunks)
BATCH = 64
for i in tqdm(range(0, len(docs), BATCH), desc="Re-embedding", unit="batch"):
    vectorstore.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])
log.info(f"Chroma rebuilt → {vectorstore._collection.count()} vectors")

# 2) BM25: rebuild from new chunks
bm25_retriever = BM25Retriever(all_chunks)

# 3) Rewire ensemble + reranker (they held refs to the OLD objects)
ensemble = EnsembleRetriever(vectorstore, bm25_retriever, fetch_k=20, weights=(1.0, 1.0))
hybrid_reranked = RerankedRetriever(ensemble, reranker, fetch_k=20)
log.info("Phase 1 pipeline rebuilt — structure-aware end to end")

# 4) Verify: citations should now show p.N and § Section
result = answer_phase1("Right of private defence against the act of a person of unsound mind",
                        retriever="hybrid_reranked")
pretty_print(result, score_label="rerank")

2026-06-01 12:51:27,117 - WARNING | rag | Rebuilding Chroma with structure-aware chunks (one-time re-embed)…
2026-06-01 12:51:27,434 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-06-01 12:51:28,828 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Re-embedding:   0%|          | 0/10 [00:00<?, ?batch/s]

2026-06-01 12:52:19,048 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:53:05,396 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:53:58,324 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:54:59,524 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:55:49,760 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:56:19,868 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:57:08,465 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:57:58,221 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 12:58:39,326 - INFO    | httpx | HTTP Request: POST http://localhost:11434/ap

Right of private defence against the act of a person of unsound mind

When an act would otherwise be a specific offence, but is committed by a person due to certain factors, every person retains the same right of private defence against that act as they would if the act were a full offence [3].

This right applies when the act is done by reason of:
*   The youth [3, 1];
*   The want of maturity of understanding [3, 1];
*   The unsoundness of mind [3, 1];
*   Intoxication [3, 1];
*   Any misconception on the part of the person [3, 1].

**Example:**
If a person of unsound mind (Z) attempts to kill another person (A), Z is guilty of no offence, but A has the same right of private defence as if Z were sane [1, 2, 5].

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_6.pdf  |  p.1   (rerank=1.000)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf  |  p.1 | § Illustrations.   (rerank=0.999)
   [3] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf  |  p.1 

In [123]:
# [████████████████████░░░░░░░░░░░░░░] ~60%

# ✓ PHASE 0 — Naive RAG            (baseline)
# ✓ PHASE 1 — Hybrid Retrieval     (DONE)
#     ✓ BM25 sparse retriever
#     ✓ Ensemble (dense ⊕ BM25, RRF fusion)
#     ✓ Cross-encoder reranker (BGE)
#     ✓ Structure-aware chunking + real page/section citations
#   → next: PHASE 2 or PHASE 4

## phase 1 and phase 2 done
## moving to phase 4 evaluation & metrics

## adding query intelligence in the next step, once the evaluation framework is in place to show its impact

In [40]:
import random


class EvalExample(BaseModel):
    """One labelled eval case. Source-path level → survives re-chunking.

    Convention: gold_source_paths == []  means  "system SHOULD refuse / find nothing"
    (used later to test the 'I don't know' path and the min_score threshold).
    """
    question: str
    gold_source_paths: list[str]
    gold_snippets: list[str] = Field(default_factory=list)
    reference_answer: str | None = None
    difficulty: Literal["easy", "medium", "hard"] = "medium"
    notes: str = ""


QGEN_PROMPT = """You are creating evaluation questions for an Indian Penal Code knowledge base.

Given the SOURCE PASSAGE below, write ONE realistic question a lawyer, student, or legal researcher would ask, where THIS passage contains the answer.

Rules:
- Answerable using ONLY facts in this passage.
- Do NOT reference "this document/passage/section/above". Ask as if you don't know where the answer lives.
- Prefer specific factual questions (section numbers, defined terms, punishments, illustrations) over vague ones.
- Also copy the SHORT exact answer snippet verbatim from the passage.

Return STRICT JSON, no markdown fences:
{{"question": "...", "answer_snippet": "...", "difficulty": "easy|medium|hard"}}

SOURCE PASSAGE:
{passage}"""


def generate_eval_set(
    chunks: list[RagChunk],
    n: int = 20,
    seed: int = 42,
    min_chunk_chars: int = 200,
) -> list[EvalExample]:
    """LLM-synthesised eval set. CURATE the output by hand afterwards — not optional."""
    rng = random.Random(seed)
    pool = [c for c in chunks if len(c.text) >= min_chunk_chars] or list(chunks)
    sample = rng.sample(pool, min(n, len(pool)))

    examples: list[EvalExample] = []
    for i, ch in enumerate(tqdm(sample, desc="Generating Q", unit="q"), 1):
        prompt = QGEN_PROMPT.format(passage=ch.text[:2000])
        try:
            resp = llm.invoke([HumanMessage(content=prompt)])
            raw = re.sub(r"^```(?:json)?|```$", "", resp.content.strip(),
                         flags=re.MULTILINE).strip()
            data = json.loads(raw)
        except Exception as e:
            log.warning(f"Q{i}: generation/parse failed ({e}); skipping")
            continue

        q = (data.get("question") or "").strip()
        snippet = (data.get("answer_snippet") or "").strip()
        diff = data.get("difficulty", "medium")
        diff = diff if diff in ("easy", "medium", "hard") else "medium"

        if len(q) < 12 or any(bad in q.lower() for bad in (
            "this document", "this passage", "the table", "above", "the text", "this section"
        )):
            log.warning(f"Q{i}: degenerate question rejected: {q!r}")
            continue

        examples.append(EvalExample(
            question=q,
            gold_source_paths=[ch.source_path],
            gold_snippets=[snippet] if snippet else [],
            difficulty=diff,
            notes=f"auto-gen from {ch.source_format} chunk {ch.chunk_id[:8]}",
        ))

    log.info(f"Generated {len(examples)} eval examples (requested {n})")
    return examples


def save_eval_set(examples: list[EvalExample], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps([e.model_dump() for e in examples], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    log.info(f"Saved {len(examples)} examples → {path.relative_to(PROJECT_ROOT)}")


def load_eval_set(path: Path) -> list[EvalExample]:
    if not path.exists():
        return []
    return [EvalExample(**d) for d in json.loads(path.read_text(encoding="utf-8"))]

### generating QnA for evaluation

In [41]:
import random
import time


EASY_PROMPT = """Create an EASY question for an Indian Penal Code knowledge base.

Direct factual lookup answerable by 1-10 words copied verbatim from the passage.
Use similar vocabulary to the passage. Ask about: section number, defined term, punishment, name, date.

Return STRICT JSON, no markdown:
{{"question": "...", "answer_snippet": "...", "reference_answer": "..."}}

PASSAGE:
{passage}"""


MEDIUM_PROMPT = """Create a MEDIUM-difficulty question for an Indian Penal Code knowledge base.

Rules:
- Paraphrase the legal terms (e.g. "stealing" not "theft", "intent to harm" not "mens rea").
- NOT answerable by keyword matching.
- 1-3 sentence answer derived from the passage.

Return STRICT JSON, no markdown:
{{"question": "...", "answer_snippet": "...", "reference_answer": "..."}}

PASSAGE:
{passage}"""


HARD_PROMPT = """Create a HARD question for an Indian Penal Code knowledge base.

You have TWO passages. Write ONE question requiring BOTH to answer.
Pick one of: COMPARISON, APPLICATION (hypothetical scenario), EXCEPTION REASONING, CROSS-REFERENCE.

Rules:
- Paraphrased legal vocabulary — do NOT echo the passages.
- Answer (2-4 sentences) MUST require facts from both passages.
- Do NOT reference "passage A/B" in the question.

Return STRICT JSON, no markdown:
{{"question": "...", "answer_snippet": "...", "reference_answer": "..."}}

PASSAGE A:
{passage_a}

PASSAGE B:
{passage_b}"""


def _normalize_q(q: str) -> str:
    return " ".join(q.lower().split())


def _parse_llm_json(text: str) -> dict | None:
    try:
        cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
        return json.loads(cleaned)
    except Exception:
        return None


_DEGENERATE_MARKERS = (
    "this document", "this passage", "passage a", "passage b",
    "this section", "above", "the text", "the table", "both passages",
)


def _build_example(
    prompt: str,
    chunk_metas: list[tuple[str, str]],
    difficulty: str,
    seen: set[str],
) -> EvalExample | None:
    resp = llm.invoke([HumanMessage(content=prompt)])
    data = _parse_llm_json(resp.content)
    if not data:
        return None

    q = (data.get("question") or "").strip()
    snippet = (data.get("answer_snippet") or "").strip()
    ref_ans = (data.get("reference_answer") or "").strip() or None

    if len(q) < 12 or any(m in q.lower() for m in _DEGENERATE_MARKERS):
        return None

    nq = _normalize_q(q)
    if nq in seen:
        return None
    seen.add(nq)

    return EvalExample(
        question=q,
        gold_source_paths=[sp for sp, _ in chunk_metas],
        gold_snippets=[snippet] if snippet else [],
        reference_answer=ref_ans,
        difficulty=difficulty,
        notes=f"auto-gen ({difficulty}) from chunks {','.join(c for _, c in chunk_metas)}",
    )


def generate_balanced_eval_set(
    chunks: list[RagChunk],
    n_easy: int = 40,
    n_medium: int = 60,
    n_hard: int = 100,
    seed: int = 42,
    min_chunk_chars: int = 200,
    max_attempts_multiplier: float = 1.6,
    save_every: int = 20,
    save_path: Path | None = None,
) -> list[EvalExample]:
    rng = random.Random(seed)
    pool = [c for c in chunks if len(c.text) >= min_chunk_chars] or list(chunks)
    if not pool:
        raise ValueError("No suitable chunks in pool")

    examples: list[EvalExample] = []
    seen: set[str] = set()

    plan = [
        ("easy",   n_easy,   EASY_PROMPT,   1),
        ("medium", n_medium, MEDIUM_PROMPT, 1),
        ("hard",   n_hard,   HARD_PROMPT,   2),
    ]

    for diff, target, template, n_chunks in plan:
        log.info(f"--- {target} {diff.upper()} questions ---")
        produced = 0
        attempts = 0
        max_attempts = int(target * max_attempts_multiplier) + target
        pbar = tqdm(total=target, desc=f"{diff:<6}", unit="q")

        while produced < target and attempts < max_attempts:
            attempts += 1
            if n_chunks == 1:
                ch = rng.choice(pool)
                prompt = template.format(passage=ch.text[:2000])
                metas = [(ch.source_path, ch.chunk_id[:8])]
            else:
                ch_a = rng.choice(pool)
                others = [c for c in pool if c.source_path != ch_a.source_path]
                ch_b = rng.choice(others) if others else rng.choice(pool)
                prompt = template.format(passage_a=ch_a.text[:1500], passage_b=ch_b.text[:1500])
                metas = [(ch_a.source_path, ch_a.chunk_id[:8]),
                         (ch_b.source_path, ch_b.chunk_id[:8])]

            try:
                ex = _build_example(prompt, metas, diff, seen)
            except Exception as e:
                log.debug(f"{diff}: error {e}")
                continue
            if ex is None:
                continue

            examples.append(ex)
            produced += 1
            pbar.update(1)

            if save_path and produced % save_every == 0:
                save_eval_set(examples, save_path)

        pbar.close()
        if produced < target:
            log.warning(f"{diff}: only {produced}/{target} after {attempts} attempts")

    if save_path:
        save_eval_set(examples, save_path)

    counts = {d: sum(1 for e in examples if e.difficulty == d) for d in ("easy", "medium", "hard")}
    log.info(f"Done. Distribution: {counts}")
    return examples

In [65]:
# Delete the stale eval first
EVAL_DIR = PROJECT_ROOT / "eval"
EVAL_SET_PATH = EVAL_DIR / "eval_set.json"
EVAL_SET_PATH.unlink(missing_ok=True)

start = time.time()
eval_set = generate_balanced_eval_set(
    chunks=all_chunks,
    n_easy=20,
    n_medium=30,
    n_hard=50,
    seed=42,
    save_every=5,
    save_path=EVAL_SET_PATH,
)
elapsed = time.time() - start
log.info(f"Total time: {elapsed/60:.1f} min")

# Eyeball samples per difficulty
from collections import Counter
print("\nDistribution:", Counter(e.difficulty for e in eval_set))
print(f"With reference answers: {sum(1 for e in eval_set if e.reference_answer)}/{len(eval_set)}")

print("\nSample per difficulty:")
for diff in ("easy", "medium", "hard"):
    sample = next((e for e in eval_set if e.difficulty == diff), None)
    if sample:
        print(f"\n[{diff.upper()}] {sample.question}")
        print(f"  ref: {(sample.reference_answer or '—')[:160]}")
        print(f"  gold sources: {sample.gold_source_paths}")

2026-05-27 12:50:29,934 - INFO    | rag | --- 20 EASY questions ---


easy  :   0%|          | 0/20 [00:00<?, ?q/s]

2026-05-27 12:50:30,521 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:50:57,735 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:51:24,827 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:51:54,362 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:52:15,895 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:52:43,704 - INFO    | rag | Saved 5 examples → eval/eval_set.json
2026-05-27 12:52:44,259 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:53:11,684 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:53:39,313 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:53

medium:   0%|          | 0/30 [00:00<?, ?q/s]

2026-05-27 12:59:00,172 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:59:28,246 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:59:45,314 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:00:14,575 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:00:41,019 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:01:07,402 - INFO    | rag | Saved 25 examples → eval/eval_set.json
2026-05-27 13:01:07,947 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:01:37,064 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:02:10,410 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:0

hard  :   0%|          | 0/50 [00:00<?, ?q/s]

2026-05-27 13:13:56,892 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:14:43,127 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:15:21,990 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:16:09,184 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:16:57,053 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:17:43,943 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:18:26,555 - INFO    | rag | Saved 55 examples → eval/eval_set.json
2026-05-27 13:18:27,103 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:19:07,824 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:1

KeyboardInterrupt: 

In [ ]:
# EVAL_DIR = PROJECT_ROOT / "eval"
# EVAL_SET_PATH = EVAL_DIR / "eval_set.json"

# REGENERATE_EVAL = True  # flip to False AFTER you've curated the set

# if REGENERATE_EVAL or not EVAL_SET_PATH.exists():
#     eval_set = generate_eval_set(all_chunks, n=20, seed=42)
#     save_eval_set(eval_set, EVAL_SET_PATH)
# else:
#     eval_set = load_eval_set(EVAL_SET_PATH)
#     log.info(f"Loaded {len(eval_set)} curated examples")

# print(f"\n{len(eval_set)} eval examples — REVIEW THESE:\n")
# for i, ex in enumerate(eval_set, 1):
#     print(f"[{i}] ({ex.difficulty}) {ex.question}")
#     print(f"     gold : {ex.gold_source_paths}")
#     print(f"     snip : {ex.gold_snippets}\n")

In [42]:
import unicodedata


def _normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"\s+", " ", text).strip()


def snippet_in_docs(snippet: str, docs: list[Document], min_overlap: float = 0.6) -> bool:
    """Gold snippet present in any retrieved doc: exact (normalized) OR token-overlap."""
    if not snippet:
        return False
    nsnip = _normalize(snippet)
    for d in docs:
        if nsnip in _normalize(d.page_content):
            return True
    snip_tokens = set(nsnip.split())
    if not snip_tokens:
        return False
    for d in docs:
        doc_tokens = set(_normalize(d.page_content).split())
        if len(snip_tokens & doc_tokens) / len(snip_tokens) >= min_overlap:
            return True
    return False


def evaluate_retriever(retrieve_fn, eval_set: list[EvalExample], k: int = 5) -> dict:
    """Hit@k, Recall@k, MRR, Snippet-Hit@k for a single retriever."""
    n = hits = snippet_hits = 0
    recall_sum = rr_sum = 0.0
    per_diff = defaultdict(lambda: {"n": 0, "hit": 0})

    for ex in eval_set:
        gold = set(ex.gold_source_paths)
        if not gold:
            continue  # negatives belong to the generation/threshold eval
        n += 1

        docs = retrieve_fn(ex.question, k)
        sources = [d.metadata.get("source_path") for d in docs]

        hit = any(s in gold for s in sources)
        hits += int(hit)

        recall_sum += len({s for s in sources if s in gold}) / len(gold)

        rr = 0.0
        for rank, s in enumerate(sources, 1):
            if s in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr

        if any(snippet_in_docs(sn, docs) for sn in ex.gold_snippets):
            snippet_hits += 1

        per_diff[ex.difficulty]["n"] += 1
        per_diff[ex.difficulty]["hit"] += int(hit)

    if n == 0:
        return {"n": 0}

    return {
        "n": n,
        "hit@k": hits / n,
        "recall@k": recall_sum / n,
        "mrr": rr_sum / n,
        "snippet_hit@k": snippet_hits / n,
        "by_difficulty": {
            d: round(v["hit"] / v["n"], 3)
            for d, v in sorted(per_diff.items()) if v["n"]
        },
    }

In [68]:
# Uniform adapters: (query, k) -> list[Document]
RETRIEVERS = {
    "dense":           lambda q, k: [d for d, _ in vectorstore.similarity_search_with_score(q, k=k)],
    "bm25":            lambda q, k: [d for d, _ in bm25_retriever.retrieve(q, k=k)],
    "hybrid":          lambda q, k: [d for d, _ in ensemble.retrieve(q, top_k=k)],
    "hybrid_reranked": lambda q, k: [d for d, _ in hybrid_reranked.retrieve(q, top_k=k)],
}

EVAL_K = cfg.TOP_K
eval_set = load_eval_set(EVAL_SET_PATH)
n_pos = len([e for e in eval_set if e.gold_source_paths])
print(f"Evaluating {n_pos} positive examples @ k={EVAL_K}\n")

results = {}
for name, fn in RETRIEVERS.items():
    log.info(f"Evaluating: {name}")
    results[name] = evaluate_retriever(fn, eval_set, k=EVAL_K)

print(f"\n{'Retriever':<18}{'Hit@k':>8}{'Recall@k':>10}{'MRR':>8}{'Snippet@k':>11}")
print("-" * 55)
for name, r in results.items():
    if r.get("n", 0) == 0:
        continue
    print(f"{name:<18}{r['hit@k']:>8.3f}{r['recall@k']:>10.3f}"
          f"{r['mrr']:>8.3f}{r['snippet_hit@k']:>11.3f}")
print("-" * 55)

print("\nHit@k by difficulty:")
for name, r in results.items():
    if r.get("n", 0):
        print(f"  {name:<18} {r['by_difficulty']}")

# Persist for regression history
RESULTS_DIR = EVAL_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
run_path = RESULTS_DIR / f"retrieval_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.json"
run_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
log.info(f"Saved run → {run_path.relative_to(PROJECT_ROOT)}")

2026-05-27 13:47:02,991 - INFO    | rag | Evaluating: dense


Evaluating 90 positive examples @ k=5



2026-05-27 13:47:04,050 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,122 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,188 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,263 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,328 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,394 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,467 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,533 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,601 - INFO    | httpx | HTTP Request: POST http://localhost:11434/ap


Retriever            Hit@k  Recall@k     MRR  Snippet@k
-------------------------------------------------------
dense                0.856     0.700   0.564      0.422
bm25                 0.744     0.594   0.519      0.389
hybrid               0.856     0.700   0.646      0.433
hybrid_reranked      0.956     0.778   0.724      0.433
-------------------------------------------------------

Hit@k by difficulty:
  dense              {'easy': 0.9, 'hard': 0.75, 'medium': 0.967}
  bm25               {'easy': 0.95, 'hard': 0.775, 'medium': 0.567}
  hybrid             {'easy': 1.0, 'hard': 0.825, 'medium': 0.8}
  hybrid_reranked    {'easy': 1.0, 'hard': 0.95, 'medium': 0.933}


In [91]:
# token evaluation
# ipc
# iso COMPLIANCE 
# US legal database

In [92]:
#pip install "ragas>=0.2" datasets nest_asyncio --break-system-packages

In [43]:
import nest_asyncio
nest_asyncio.apply()  # let RAGAS's async run inside Jupyter's event loop

from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)
from ragas.run_config import RunConfig

/tmp/ipykernel_114760/439603412.py:7: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_114760/439603412.py:7: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
/tmp/ipykernel_114760/439603412.py:7: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import (


In [98]:

import random
from collections import Counter
eval_set = [EvalExample(**d) for d in json.loads(EVAL_SET_PATH.read_text(encoding="utf-8"))]
log.info(f"Reloaded {len(eval_set)} EvalExample objects")
# Keep eval_set as EvalExample objects (the schema-aware version)
positives = [e for e in eval_set if e.gold_source_paths]

# Stratified sample so all difficulties are represented
rng = random.Random(42)
by_diff = {"easy": [], "medium": [], "hard": []}
for e in positives:
    by_diff[e.difficulty].append(e)

sample = (
    rng.sample(by_diff["easy"],   min(8,  len(by_diff["easy"])))
  + rng.sample(by_diff["medium"], min(12, len(by_diff["medium"])))
  + rng.sample(by_diff["hard"],   min(20, len(by_diff["hard"])))
)
log.info(f"RAGAS sample: {Counter(e.difficulty for e in sample)}")

ragas_rows = []
for ex in tqdm(sample, desc="Generating answers"):
    # Retrieval — same path as production answer pipeline
    results = hybrid_reranked.retrieve(ex.question, top_k=cfg.TOP_K)
    contexts = [d.page_content for d, _ in results]

    # ✅ Use build_context (numbered citations, provenance)
    context_block, _ = build_context(results)
    user_msg = f"CONTEXT:\n{context_block}\n\nQUESTION: {ex.question}\n\nANSWER:"

    # ✅ Pass SYSTEM_PROMPT via SystemMessage
    resp = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ])

    ragas_rows.append({
        "user_input": ex.question,
        "retrieved_contexts": contexts,
        "response": resp.content,
        # Prefer the rough reference_answer over a snippet — better signal for the judge
        "reference": ex.reference_answer or (ex.gold_snippets[0] if ex.gold_snippets else ""),
    })

ragas_dataset = EvaluationDataset.from_list(ragas_rows)
log.info(f"RAGAS dataset built: {len(ragas_dataset)} rows")


2026-05-27 22:05:45,477 - INFO    | rag | Reloaded 90 EvalExample objects
2026-05-27 22:05:45,478 - INFO    | rag | RAGAS sample: Counter({'hard': 20, 'medium': 12, 'easy': 8})
Generating answers:   0%|          | 0/40 [00:00<?, ?it/s]2026-05-27 22:05:46,594 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:05:56,006 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Generating answers:   2%|▎         | 1/40 [00:30<19:33, 30.09s/it]2026-05-27 22:06:16,561 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:06:18,832 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Generating answers:   5%|▌         | 2/40 [00:52<16:15, 25.67s/it]2026-05-27 22:06:38,227 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:06:40,473 - INFO    | httpx | HTTP Request: POST ht

In [44]:
metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    SemanticSimilarity(embeddings=ragas_emb),
    # AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb),  # uncomment for overnight run
]

NameError: name 'ragas_llm' is not defined

In [128]:
IPC_NEGATIVES = [
    # Obvious negatives (different statutes)
    {
        "question": "What is the procedure for filing an income tax return in India?",
        "gold_source_paths": [], "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "easy",
        "notes": "NEGATIVE: tax law, not IPC",
    },
    {
        "question": "What is the GST rate on legal services in India?",
        "gold_source_paths": [], "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "easy",
        "notes": "NEGATIVE: indirect tax, not IPC",
    },
    # Adversarial (high keyword overlap, sounds IPC-adjacent, isn't)
    {
        "question": "What is the procedure for filing an FIR under the Code of Criminal Procedure?",
        "gold_source_paths": [], "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "hard",
        "notes": "ADVERSARIAL: CrPC procedure, not IPC substantive law — heavy overlap on 'criminal'/'procedure'",
    },
    {
        "question": "What are the provisions for cybercrime under the Information Technology Act 2000?",
        "gold_source_paths": [], "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "hard",
        "notes": "ADVERSARIAL: IT Act, not IPC — overlaps on 'offence'/'punishment'",
    },
    {
        "question": "How does the Indian Evidence Act define hearsay evidence?",
        "gold_source_paths": [], "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "hard",
        "notes": "ADVERSARIAL: Evidence Act, not IPC",
    },
    {
        "question": "What does Article 21 of the Indian Constitution guarantee?",
        "gold_source_paths": [], "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "medium",
        "notes": "ADVERSARIAL: Constitutional law, not IPC",
    },
]


# Merge into eval set (dedupe by normalized question)
# Robust load — bypass any overridden load_eval_set
existing = [EvalExample(**d) for d in json.loads(EVAL_SET_PATH.read_text(encoding="utf-8"))]
seen = {e.question.strip().lower() for e in existing}
added = 0
for d in IPC_NEGATIVES:
    if d['question'].strip().lower() in seen:
        continue
    existing.append(EvalExample(**d))
    added += 1
save_eval_set(existing, EVAL_SET_PATH)
eval_set = existing
log.info(f"Added {added} negatives → {len(eval_set)} total "
         f"({sum(1 for e in eval_set if not e.gold_source_paths)} negatives)")

2026-06-01 13:03:24,319 - INFO    | rag | Saved 96 examples → eval/eval_set.json
2026-06-01 13:03:24,319 - INFO    | rag | Added 0 negatives → 96 total (6 negatives)


In [104]:
REFUSAL_MARKERS = (
    "i don't have that information",
    "i do not have that information",
    "not in the provided documents",
    "not mentioned in",
    "the context does not",
    "no information about",
    "cannot find",
    "i cannot answer",
)

def is_refusal(text: str) -> bool:
    t = text.lower()
    return any(m in t for m in REFUSAL_MARKERS)


negatives = [e for e in eval_set if not e.gold_source_paths]
print(f"Testing {len(negatives)} negatives with NO min_score (raw pipeline)\n")

negative_results = []
refused = 0
for ex in tqdm(negatives, desc="Negatives"):
    results = hybrid_reranked.retrieve(ex.question, top_k=cfg.TOP_K)
    top_score = results[0][1] if results else 0.0   # reranker score of top chunk
    context_block, _ = build_context(results)
    user_msg = f"CONTEXT:\n{context_block}\n\nQUESTION: {ex.question}\n\nANSWER:"
    resp = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ])
    refusal = is_refusal(resp.content)
    refused += int(refusal)
    negative_results.append({
        "q": ex.question[:60],
        "difficulty": ex.difficulty,
        "top_rerank": top_score,
        "refused": refusal,
        "answer_preview": resp.content[:120].replace("\n", " "),
    })

print(f"\nRefusal rate: {refused}/{len(negatives)} = {refused/len(negatives):.0%}\n")
print(f"{'Q':<62}{'diff':<8}{'top':<8}{'refused':<8}")
for r in negative_results:
    print(f"{r['q']:<62}{r['difficulty']:<8}{r['top_rerank']:.3f}   {'✓' if r['refused'] else '✗'}")
print("\nFailed answers (would-be confabulations):")
for r in negative_results:
    if not r['refused']:
        print(f"  • {r['q']}")
        print(f"    → {r['answer_preview']!r}")

Testing 6 negatives with NO min_score (raw pipeline)



Negatives:   0%|          | 0/6 [00:00<?, ?it/s]2026-05-27 22:39:48,257 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:39:52,537 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Negatives:  17%|█▋        | 1/6 [00:17<01:28, 17.62s/it]2026-05-27 22:40:05,785 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:40:07,935 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Negatives:  33%|███▎      | 2/6 [00:34<01:08, 17.16s/it]2026-05-27 22:40:21,689 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:40:24,076 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Negatives:  50%|█████     | 3/6 [00:51<00:51, 17.17s/it]2026-05-27 22:40:38,881 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1


Refusal rate: 6/6 = 100%

Q                                                             diff    top     refused 
What is the procedure for filing an income tax return in Ind  easy    0.002   ✓
What is the GST rate on legal services in India?              easy    0.003   ✓
What is the procedure for filing an FIR under the Code of Cr  hard    0.136   ✓
What are the provisions for cybercrime under the Information  hard    0.417   ✓
How does the Indian Evidence Act define hearsay evidence?     hard    0.007   ✓
What does Article 21 of the Indian Constitution guarantee?    medium  0.367   ✓

Failed answers (would-be confabulations):


In [134]:
# Wire min_score into the production retriever
hybrid_reranked = RerankedRetriever(
    base_retriever=ensemble,
    reranker=reranker,
    fetch_k=20,
    min_score=0.5,   # below this → empty context → forces refusal
)
log.info(f"Production retriever: hybrid+rerank with min_score={hybrid_reranked.min_score}")

# Quick re-verify on the negatives
refused_after = 0
for ex in negatives:
    results = hybrid_reranked.retrieve(ex.question, top_k=cfg.TOP_K)
    if not results:
        refused_after += 1   # empty context → answer pipeline will refuse
        continue
    context_block, _ = build_context(results)
    resp = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"CONTEXT:\n{context_block}\n\nQUESTION: {ex.question}\n\nANSWER:"),
    ])
    refused_after += int(is_refusal(resp.content))

print(f"Refusal rate with min_score=0.5: {refused_after}/{len(negatives)}")

2026-06-01 13:04:49,466 - INFO    | rag | Production retriever: hybrid+rerank with min_score=0.5


NameError: name 'negatives' is not defined

In [ ]:
from datetime import datetime, timezone
import pandas as pd

# Make sure the results directory exists
RESULTS_DIR = EVAL_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running RAGAS over {len(ragas_dataset)} rows. "
      f"Plan ~1–2 min/row on local Gemma.\n")

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    show_progress=True,
)

df = result.to_pandas()

# Numeric metric columns (avoid hardcoding names — they shift across RAGAS versions)
metric_cols = list(df.select_dtypes(include="number").columns)

print("\n=== Per-row scores ===")
print(df[["user_input"] + metric_cols].to_string(index=False, max_colwidth=55))

print("\n=== Aggregate ===")
for col in metric_cols:
    print(f"  {col:<50} {df[col].mean():.3f}")

# Save CSV (with timestamp so multiple runs don't overwrite)
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
result_path = RESULTS_DIR / f"ragas_{ts}.csv"
df.to_csv(result_path, index=False)
log.info(f"Saved → {result_path.relative_to(PROJECT_ROOT)}")

In [107]:
print(f"Dataset size: {len(ragas_dataset)} rows")
print(f"Metrics: {[type(m).__name__ for m in metrics]}")

Dataset size: 40 rows
Metrics: ['Faithfulness', 'ResponseRelevancy', 'LLMContextPrecisionWithoutReference', 'SemanticSimilarity']


In [108]:
from datetime import datetime, timezone

RESULTS_DIR = EVAL_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running RAGAS over {len(ragas_dataset)} rows. Plan ~1–2 min/row on local Gemma.\n")

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    show_progress=True,
)

df = result.to_pandas()
metric_cols = list(df.select_dtypes(include="number").columns)

print("\n=== Per-row ===")
print(df[["user_input"] + metric_cols].to_string(index=False, max_colwidth=55))

print("\n=== Aggregate ===")
for col in metric_cols:
    print(f"  {col:<50} {df[col].mean():.3f}")

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
result_path = RESULTS_DIR / f"ragas_{ts}.csv"
df.to_csv(result_path, index=False)
log.info(f"Saved → {result_path.relative_to(PROJECT_ROOT)}")

Running RAGAS over 40 rows. Plan ~1–2 min/row on local Gemma.



Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

2026-05-27 22:52:47,100 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 22:53:34,281 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 22:54:24,822 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 22:54:37,304 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 22:54:48,309 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 22:55:22,009 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:55:32,112 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 22:55:43,187 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 22:56:19,519 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat


=== Per-row ===
                                             user_input  faithfulness  answer_relevancy  llm_context_precision_without_reference  semantic_similarity
What is the maximum imprisonment period for cruelty ...      1.000000          0.911654                                      NaN             0.658246
According to Section 283, what is the fine for dange...      0.500000          0.994102                                      NaN             0.511915
According to Section 228, what is the maximum fine f...      1.000000          0.556156                                      NaN             0.224534
According to Section 33, what must the harm be for i...      1.000000          0.510664                                      1.0             0.274839
What is the maximum term of imprisonment for wearing...      1.000000          0.997011                                      NaN             0.392323
According to Section 109, what is the rule regarding...      0.750000          0.63

2026-05-28 03:43:40,695 - INFO    | rag | Saved → eval/results/ragas_20260527_194340.csv


In [106]:
# [████████████████████████░░░░░░░░░░] ~70%

# ✓ PHASE 0 — Naive RAG
# ✓ PHASE 1 — Hybrid Retrieval
# ✓ PHASE 4 — Eval & Observability      (CLOSED)
#     ✓ Retrieval metrics (Hit@k 0.956, MRR 0.724, hybrid_reranked wins)
#     ✓ RAGAS generation metrics (Faithfulness/Relevancy/ContextPrecision)
#     ✓ Refusal rate 100% on adversarial negatives
#     ✓ min_score=0.5 set, defense in depth
#   → PHASE 2 — Query Intelligence ◄── NEXT
#     PHASE 3 — Agentic Orchestration
#     PHASE 5 — Production Hardening
#     PHASE 6 — Cloud & Scale

In [109]:
## starting phase 2 now
## query intelligence: reformulation, decomposition, retrieval-time generation (RAG+), etc.

In [45]:
LEGAL_QUERY_EXPAND_PROMPT = """You are an expert in Indian criminal law helping search the Indian Penal Code (IPC).

Given the USER QUESTION below, generate {n} alternative search queries that maximize the chance of retrieving the relevant IPC sections.

Each variant must serve a different retrieval mode:
1. **Normalized**: formal legal vocabulary (e.g., "stealing" → "theft / dishonest taking", "punishment" → "imprisonment / fine / penalty").
2. **Keyword-extraction**: just the distinctive nouns and offense terms — short, dense, no filler words.
3. **Paraphrase**: same meaning, different vocabulary.
4. **Section-anchored**: if a section number or named offense is mentioned, lead with it; otherwise reframe as "section relating to ...".

Preserve all specifics: section numbers, fine amounts, time periods, named offenses.

Return STRICT JSON, no markdown:
{{"variants": ["...", "...", "...", "..."]}}

USER QUESTION:
{query}"""


class MultiQueryRetriever:
    """Expand a query into N variants → retrieve each → fuse via RRF.

    Wraps ANY base retriever (e.g., hybrid_reranked). The base retriever
    does the document lookup; this class only handles expansion + fusion.
    """

    def __init__(
        self,
        base_retriever,
        llm,
        n_variants: int = 3,
        rrf_k: int = 60,
        per_variant_fetch: int = 10,
    ):
        self.base = base_retriever
        self.llm = llm
        self.n_variants = n_variants
        self.rrf_k = rrf_k
        self.per_variant_fetch = per_variant_fetch

    def expand(self, query: str) -> list[str]:
        """LLM-generate N variants. Always returns [original, ...N variants]."""
        prompt = LEGAL_QUERY_EXPAND_PROMPT.format(n=self.n_variants, query=query)
        try:
            resp = self.llm.invoke([HumanMessage(content=prompt)])
            cleaned = re.sub(r"^```(?:json)?|```$", "", resp.content.strip(),
                             flags=re.MULTILINE).strip()
            data = json.loads(cleaned)
            variants = [v.strip() for v in data.get("variants", [])
                        if v and isinstance(v, str)]
        except Exception as e:
            log.warning(f"[MQ] expansion failed ({e}); falling back to single query")
            variants = []
        # Always include original; dedupe by normalized form
        seen = {query.strip().lower()}
        out = [query]
        for v in variants:
            if v.strip().lower() not in seen:
                out.append(v)
                seen.add(v.strip().lower())
        return out[:self.n_variants + 1]

    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Document, float]]:
        variants = self.expand(query)
        log.info(f"[MQ] {len(variants)} variants for '{query[:60]}…'")

        rankings: list[list[Document]] = []
        for v in variants:
            results = self.base.retrieve(v, top_k=self.per_variant_fetch)
            rankings.append([d for d, _ in results])

        # Reuse the RRF fusion from Phase 1 (Cell 19)
        fused = reciprocal_rank_fusion(rankings, k=self.rrf_k)
        return fused[:top_k]

In [136]:
mq_retriever = MultiQueryRetriever(
    base_retriever=hybrid_reranked,
    llm=llm,
    n_variants=3,           # 3 variants + original = 4 retrievals total
    rrf_k=60,
    per_variant_fetch=10,
)


def answer_phase2(
    query: str,
    k: int | None = None,
    retriever: str = "multi_query",
) -> dict:
    """Phase 2 answer pipeline. Adds 'multi_query' to the retriever options."""
    k = k or cfg.TOP_K

    if retriever == "hybrid_reranked":
        results = hybrid_reranked.retrieve(query, top_k=k)
    elif retriever == "multi_query":
        results = mq_retriever.retrieve(query, top_k=cfg.TOP_K)
    else:
        # delegate to phase1 for dense/bm25/hybrid
        return answer_phase1(query, k=k, retriever=retriever)

    if not results:
        return {"question": query, "answer": "No relevant docs.", "citations": [], "retriever": retriever}

    context_block, citations = build_context(results)
    user_msg = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"
    resp = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ])
    return {
        "question": query,
        "answer": resp.content,
        "citations": citations,
        "retriever": retriever,
    }


# Pick a question that scored 0.0 relevancy in your Phase 4 RAGAS run
HARD_QUERY = (
    "IPC liability for abetment of crime committed outside India (extraterritorial jurisdiction)"
)

# Show what the LLM expanded the query into
print("=" * 75)
print("QUERY EXPANSION")
print("=" * 75)
for i, v in enumerate(mq_retriever.expand(HARD_QUERY)):
    tag = "  ← original" if i == 0 else ""
    print(f"  {i}. {v}{tag}")

# A/B compare
print("\n" + "=" * 75)
print("RETRIEVER: hybrid_reranked  (Phase 1 — what we had)")
print("=" * 75)
pretty_print(answer_phase2(HARD_QUERY, retriever="hybrid_reranked"), score_label="rerank")

print("\n" + "=" * 75)
print("RETRIEVER: multi_query  (Phase 2 — new)")
print("=" * 75)
pretty_print(answer_phase2(HARD_QUERY, retriever="multi_query"), score_label="rrf")

QUERY EXPANSION


2026-06-01 13:05:48,355 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


  0. IPC liability for abetment of crime committed outside India (extraterritorial jurisdiction)  ← original
  1. Jurisdiction of IPC regarding abetment of extraterritorial crimes
  2. IPC abetment extraterritorial jurisdiction
  3. IPC provisions governing liability for instigating offenses committed in foreign territory

RETRIEVER: hybrid_reranked  (Phase 1 — what we had)


2026-06-01 13:07:46,887 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:08:45,206 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


IPC liability for abetment of crime committed outside India (extraterritorial jurisdiction)

A person is liable for abetment of an offense committed outside India if they commit the act while in India [3].

According to the law:

*   **Abetment in India of offences outside India:** A person abets an offense who, while in India, abets the commission of any act outside and beyond India that would constitute an offense if committed in India [3].
*   **Liability:** If a person in India instigates a foreigner in a foreign country to commit a crime, that person is guilty of abetting the crime [1, 3].

*(Note: The context also addresses the reverse scenario, "Abetment outside India for offence in India," where a person outside India instigates an act in India [3].)*

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_7.pdf  |  p.1 | § Illustration   (rerank=0.955)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf  |  p.2 | § of abetment   (rerank=0.895)
   [3] /h

2026-06-01 13:11:02,339 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 13:12:52,224 - INFO    | rag | [MQ] 4 variants for 'IPC liability for abetment of crime committed outside India …'
2026-06-01 13:12:53,076 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:13:11,441 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:13:29,751 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:13:47,279 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:14:43,737 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


IPC liability for abetment of crime committed outside India (extraterritorial jurisdiction)

A person is liable for abetment of an offense committed outside India if they commit the act while in India [3].

According to the law:

*   **Abetment in India of offences outside India:** A person abets an offense who, while in India, abets the commission of any act outside and beyond India that would constitute an offense if committed in India [3].
*   **Liability:** If a person in India instigates a foreigner in a foreign country to commit a crime, that person is guilty of abetting the crime [1, 3].

*(Note: The context also addresses the reverse scenario, "Abetment outside India for offence in India," where a person outside India instigates an act in India [3].)*

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_7.pdf  |  p.1 | § Illustration   (rrf=0.016)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf  |  p.2 | § of abetment   (rrf=0.016)
   [3] /home/th

In [137]:
# ─────────────────────────────────────────────────────────────────────
# Cell 32b — Phase 2 RAGAS dataset build (multi_query retriever)
# Same logic as Cell 32, only the retriever line changed.
# ─────────────────────────────────────────────────────────────────────
import random
from collections import Counter

EVAL_DIR = PROJECT_ROOT / "eval"
EVAL_SET_PATH = EVAL_DIR / "eval_set.json"
eval_set = [EvalExample(**d) for d in json.loads(EVAL_SET_PATH.read_text(encoding="utf-8"))]
log.info(f"Reloaded {len(eval_set)} EvalExample objects")
positives = [e for e in eval_set if e.gold_source_paths]

# Stratified sample — same seed → same 40 rows as Phase 4 baseline
rng = random.Random(42)
by_diff = {"easy": [], "medium": [], "hard": []}
for e in positives:
    by_diff[e.difficulty].append(e)

sample = (
    rng.sample(by_diff["easy"],   min(8,  len(by_diff["easy"])))
  + rng.sample(by_diff["medium"], min(12, len(by_diff["medium"])))
  + rng.sample(by_diff["hard"],   min(20, len(by_diff["hard"])))
)
log.info(f"RAGAS sample (Phase 2): {Counter(e.difficulty for e in sample)}")

ragas_rows_mq = []   # ← new variable so the Phase 4 one isn't clobbered
for ex in tqdm(sample, desc="Generating answers (multi_query)"):
    # ↓ ONLY CHANGE: mq_retriever instead of hybrid_reranked
    results = mq_retriever.retrieve(ex.question, top_k=cfg.TOP_K)
    contexts = [d.page_content for d, _ in results]

    context_block, _ = build_context(results)
    user_msg = f"CONTEXT:\n{context_block}\n\nQUESTION: {ex.question}\n\nANSWER:"

    resp = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ])

    ragas_rows_mq.append({
        "user_input": ex.question,
        "retrieved_contexts": contexts,
        "response": resp.content,
        "reference": ex.reference_answer or (ex.gold_snippets[0] if ex.gold_snippets else ""),
    })

ragas_dataset_mq = EvaluationDataset.from_list(ragas_rows_mq)
log.info(f"RAGAS dataset (multi_query) built: {len(ragas_dataset_mq)} rows")

2026-06-01 13:17:08,053 - INFO    | rag | Reloaded 96 EvalExample objects
2026-06-01 13:17:08,055 - INFO    | rag | RAGAS sample (Phase 2): Counter({'hard': 20, 'medium': 12, 'easy': 8})


Generating answers (multi_query):   0%|          | 0/40 [00:00<?, ?it/s]

2026-06-01 13:17:14,056 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 13:19:00,563 - INFO    | rag | [MQ] 4 variants for 'What is the maximum imprisonment period for cruelty under Se…'
2026-06-01 13:19:01,388 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:19:19,266 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:19:37,983 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:19:56,484 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 13:20:33,004 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 13:21:37,187 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 13:24:04,316 - INFO    | rag | [MQ] 4 variants for 'According to Sec

In [47]:
# ─────────────────────────────────────────────────────────────────────
# Cell 31 — RAGAS setup (judge LLM + embeddings + metrics + config)
# ─────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas")

import nest_asyncio
nest_asyncio.apply()

from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
    SemanticSimilarity,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Wrap our existing Ollama LLM + embeddings as RAGAS judges
ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    SemanticSimilarity(embeddings=ragas_emb),
]

run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=60,
    max_workers=1,     # local Ollama = one at a time
)

log.info(f"RAGAS configured: {len(metrics)} metrics, judge={cfg.OLLAMA_MODEL}")

/tmp/ipykernel_114760/3419639164.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_114760/3419639164.py:11: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
/tmp/ipykernel_114760/3419639164.py:11: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import (
/tmp/ipykernel_114760/3419639164.py:11: DeprecationWarning: Importing SemanticSimilarity 

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 33b — Phase 2 RAGAS scoring (multi_query retriever)
# ─────────────────────────────────────────────────────────────────────
from datetime import datetime, timezone
metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    SemanticSimilarity(embeddings=ragas_emb),
    # AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb),  # uncomment for overnight run
]
result_mq = evaluate(
    dataset=ragas_dataset_mq,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    show_progress=True,
)
df_mq = result_mq.to_pandas()

numeric_cols = df_mq.select_dtypes(include="number").columns.tolist()
print("\n=== Per-row (Phase 2 / multi_query) ===")
print(df_mq[["user_input"] + numeric_cols].to_string(index=False, max_colwidth=55))

print("\n=== Aggregate (Phase 2 / multi_query) ===")
for c in numeric_cols:
    print(f"  {c:<40s} {df_mq[c].mean():.3f}")

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
out_path = cfg.PROJECT_ROOT / "eval" / "results" / f"ragas_mq_{ts}.csv"
df_mq.to_csv(out_path, index=False)
log.info(f"Saved → {out_path.relative_to(cfg.PROJECT_ROOT)}")

Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

2026-06-01 17:21:00,362 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 17:23:35,691 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 17:23:49,110 - ERROR   | ragas.executor | Exception raised in Job[0]: TimeoutError()
2026-06-01 17:23:58,619 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 17:24:49,649 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 17:25:35,991 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 17:26:35,233 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 17:26:45,409 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 17:27:19,778 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK

KeyboardInterrupt: 

2026-06-01 20:14:03,947 - ERROR   | ragas.executor | Exception raised in Job[76]: TimeoutError()
2026-06-01 20:14:11,927 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 20:14:54,053 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 20:15:41,205 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-01 20:16:47,165 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 20:16:57,312 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 20:17:07,453 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 20:17:17,560 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-01 20:17:35,760 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200

In [48]:
PROJECT_ROOT

PosixPath('/home/thimu/github_vs/protoRAG/rag-pipeline')

In [149]:
# # Save the dataset cache (run this ONCE, immediately)
# import json

# RAGAS_ROWS_CACHE = PROJECT_ROOT / "data" / "processed" / "ragas_rows_mq.json"
# RAGAS_ROWS_CACHE.parent.mkdir(parents=True, exist_ok=True)

# with open(RAGAS_ROWS_CACHE, "w", encoding="utf-8") as f:
#     json.dump(ragas_rows_mq, f, indent=2, ensure_ascii=False)

# log.info(f"Cached {len(ragas_rows_mq)} ragas rows → {RAGAS_ROWS_CACHE.relative_to(PROJECT_ROOT)}")

In [49]:
# ─────────────────────────────────────────────────────────────────────
# Cell 33b — Phase 2 RAGAS scoring
#   - Loads dataset from disk (ragas_rows_mq.json — your saved Cell 32b output)
#   - Batched: 5 rows per evaluate() call
#   - Checkpointed: saves CSV after every batch, never loses progress
#   - Resumable: re-running picks up where it stopped
# ─────────────────────────────────────────────────────────────────────
import json
import pandas as pd
from datetime import datetime, timezone

# 1) Load the preserved dataset
RAGAS_ROWS_CACHE = PROJECT_ROOT / "data" / "processed" / "ragas_rows_mq.json"
ragas_rows_mq = json.loads(RAGAS_ROWS_CACHE.read_text(encoding="utf-8"))
log.info(f"Loaded {len(ragas_rows_mq)} rows from {RAGAS_ROWS_CACHE.name}")

# 2) Faster metric set — dropped LLMContextPrecisionWithoutReference (mostly NaN on Gemma)
metrics_fast = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    SemanticSimilarity(embeddings=ragas_emb),
]

BATCH_SIZE = 5
RESULTS_DIR = PROJECT_ROOT / "eval" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RESULTS_DIR / "ragas_mq_checkpoint.csv"

# 3) Resume from checkpoint if present
all_dfs: list[pd.DataFrame] = []
start_idx = 0
if CHECKPOINT_PATH.exists():
    existing = pd.read_csv(CHECKPOINT_PATH)
    all_dfs.append(existing)
    start_idx = len(existing)
    log.info(f"↻ Resuming: {start_idx} rows already scored in checkpoint")
else:
    log.info("▶ Starting fresh (no checkpoint found)")

total = len(ragas_rows_mq)

# 4) Score in batches, checkpoint after each
for batch_start in range(start_idx, total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total)
    batch_rows = ragas_rows_mq[batch_start:batch_end]

    log.info(f"▶ Scoring rows {batch_start + 1}–{batch_end} of {total}")
    batch_dataset = EvaluationDataset.from_list(batch_rows)

    result = evaluate(
        dataset=batch_dataset,
        metrics=metrics_fast,
        llm=ragas_llm,
        embeddings=ragas_emb,
        run_config=run_config,
        show_progress=False,
    )
    batch_df = result.to_pandas()
    all_dfs.append(batch_df)

    # Save checkpoint
    pd.concat(all_dfs, ignore_index=True).to_csv(CHECKPOINT_PATH, index=False)
    done = sum(len(d) for d in all_dfs)
    log.info(f"  ✓ Checkpoint saved ({done}/{total} rows on disk)")

# 5) Final aggregate + timestamped CSV
df_mq = pd.concat(all_dfs, ignore_index=True)
numeric_cols = df_mq.select_dtypes(include="number").columns.tolist()

print("\n=== Aggregate (Phase 2 / multi_query) ===")
for c in numeric_cols:
    print(f"  {c:<35s} {df_mq[c].mean():.3f}")

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
final_path = RESULTS_DIR / f"ragas_mq_{ts}.csv"
df_mq.to_csv(final_path, index=False)
log.info(f"✅ Final CSV → {final_path.relative_to(PROJECT_ROOT)}")

2026-06-02 01:30:39,790 - INFO    | rag | Loaded 40 rows from ragas_rows_mq.json
2026-06-02 01:30:39,794 - INFO    | rag | ↻ Resuming: 20 rows already scored in checkpoint
2026-06-02 01:30:39,794 - INFO    | rag | ▶ Scoring rows 21–25 of 40
2026-06-02 01:31:02,076 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-02 01:32:34,832 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-02 01:33:49,939 - ERROR   | ragas.executor | Exception raised in Job[0]: TimeoutError()
2026-06-02 01:33:58,053 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-02 01:34:50,109 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-02 01:35:33,166 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-02 01:37:02,815 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "H


=== Aggregate (Phase 2 / multi_query) ===
  faithfulness                        0.000
  answer_relevancy                    0.357
  semantic_similarity                 0.483


In [51]:
import pandas as pd

df_mq = pd.read_csv(PROJECT_ROOT / "eval" / "results" / "ragas_mq_20260601_193919.csv")

print(f"Total rows: {len(df_mq)}\n")

print("=== Per-metric distribution ===")
for col in ["faithfulness", "answer_relevancy", "semantic_similarity"]:
    s = df_mq[col]
    print(f"\n{col}:")
    print(f"  NaN count:     {s.isna().sum()}")
    print(f"  zero count:    {(s == 0).sum()}")
    print(f"  nonzero count: {((s > 0) & ~s.isna()).sum()}")
    print(f"  mean(skipna):  {s.mean():.3f}")
    print(f"  min: {s.min():.3f}  max: {s.max():.3f}")

print("\n=== Rows where faithfulness > 0 ===")
mask = df_mq["faithfulness"] > 0
print(df_mq.loc[mask, ["user_input", "faithfulness", "answer_relevancy"]].to_string(max_colwidth=60))

Total rows: 40

=== Per-metric distribution ===

faithfulness:
  NaN count:     32
  zero count:    8
  nonzero count: 0
  mean(skipna):  0.000
  min: 0.000  max: 0.000

answer_relevancy:
  NaN count:     19
  zero count:    10
  nonzero count: 11
  mean(skipna):  0.357
  min: 0.000  max: 0.992

semantic_similarity:
  NaN count:     0
  zero count:    0
  nonzero count: 40
  mean(skipna):  0.483
  min: 0.158  max: 0.930

=== Rows where faithfulness > 0 ===
Empty DataFrame
Columns: [user_input, faithfulness, answer_relevancy]
Index: []


In [52]:
refusal_mask = df_mq["response"].str.contains("don't have that information", case=False, na=False)
empty_ctx_mask = df_mq["retrieved_contexts"].astype(str).isin(["[]", "['']"])

print(f"Rows with refusal response:    {refusal_mask.sum()}")
print(f"Rows with empty contexts:      {empty_ctx_mask.sum()}")
print(f"Rows where both:                {(refusal_mask & empty_ctx_mask).sum()}")
print(f"\nRows scored 0 on relevancy that are refusals: {((df_mq['answer_relevancy']==0) & refusal_mask).sum()}/10")

Rows with refusal response:    15
Rows with empty contexts:      8
Rows where both:                8

Rows scored 0 on relevancy that are refusals: 10/10
